In [1]:
import os
import sys
import subprocess
import glob

# 🎛️ SET THIS TO TRUE FOR TPU, FALSE FOR GPU
FORCE_TPU = True

def repair_environment():

    if FORCE_TPU:
        print("🔍 Starting High-Speed TPU Repair...")

        # 1. Faster Uninstallation
        print("🧹 Wiping libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torch_xla", "torchvision", "numpy", "tensorflow", "huggingface_hub"],
                       capture_output=True)

        # 2. Parallel/Bulk Installation
        print("📥 Installing Synced TPU Stack...")
        common_args = ["install", "-q", "--no-warn-script-location"]

        if FORCE_TPU or glob.glob("/dev/accel*"):
            cmd = [
                sys.executable, "-m", "pip", *common_args,
                "torch==2.8.0",
                "torchvision==0.23.0",
                "torch_xla[tpu]==2.8.0",
                "numpy", "pyarrow==16.1.0", "fsspec", # <-- Pinned pyarrow here
                "protobuf>=5.28.0",
                "datasets>=2.20.0", "transformers", "huggingface_hub>=0.28.0", "wandb", # <-- Added >=2.20.0 to datasets
                "cloud-tpu-client", "scikit-learn", "pandas<3.0.0",
                "-f", "https://storage.googleapis.com/libtpu-releases/index.html",
                "--extra-index-url", "https://download.pytorch.org/whl/cpu"
            ]
            subprocess.check_call(cmd)
        else:
            # Fallback
            cmd = [
                sys.executable, "-m", "pip", *common_args, "-U",
                "torch", "datasets", "pyarrow", "transformers", "huggingface_hub>=0.28.0", "fsspec", "wandb", "scipy", "numpy", "pandas<3.0.0"
            ]
            subprocess.check_call(cmd)

        print("\n✅ TPU REPAIR COMPLETE.")
        print("⚠️ Click 'Run' -> 'Restart Session' NOW.")

    else:
        print("🔍 Starting Robust GPU Repair...")

        # 1. Clean Wipe
        print("🧹 Wiping conflicting libraries...")
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q",
                        "torch", "torchvision", "torchaudio", "huggingface_hub"],
                       capture_output=True)

        # 2. Setup Arguments
        common_args = ["install", "-q", "--no-warn-script-location"]

        try:
            print("📥 Installing GPU/CUDA Stack...")

            print("   ⚡ Part 1: PyTorch Core...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args,
                "torch", "torchvision", "torchaudio",
                "--index-url", "https://download.pytorch.org/whl/cu121"
            ])

            print("   ⚡ Part 2: Transformers & Data...")
            subprocess.check_call([
                sys.executable, "-m", "pip", *common_args, "-U",
                "datasets", "transformers", "huggingface_hub>=0.28.0",
                "wandb", "pandas<3.0.0"
            ])

            print("\n✅ GPU REPAIR COMPLETE.")
            print("⚠️ MANDATORY: Click 'Run' -> 'Restart Session' NOW.")

        except subprocess.CalledProcessError as e:
            print(f"\n❌ Installation failed. Error: {e}")
            print("💡 Try manually restarting the session and running this cell again.")

if __name__ == "__main__":
    repair_environment()

🔍 Starting High-Speed TPU Repair...
🧹 Wiping libraries...
📥 Installing Synced TPU Stack...



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip



✅ TPU REPAIR COMPLETE.
⚠️ Click 'Run' -> 'Restart Session' NOW.


In [2]:
%%writefile model.py

##################################################
# Defines HELM Phase 13A: Elastic Threshold Router
# Has a total of 32 heads, d_head = 64; only 16 will be used at a time
# True Decoupling of d_model = d_head * n_head
# Acheived via expansion layer
# Target 8 16 32
# 4 Perm heads
# No Dead Head Penalty
# Still Use Clamp
##################################################

import os
import json
import torch
import numpy as np
from safetensors.torch import load_file
import math
from math import sqrt
import random
import torch.nn.functional as F
import torch.nn as nn
try:
    from torch_xla.utils.checkpoint import checkpoint as _xla_checkpoint
except Exception:
    _xla_checkpoint = None
from transformers import AutoTokenizer
from transformers import PretrainedConfig, PreTrainedModel



# modified justnorm() function
# better than F.normalize(), max() causes micro walls during gradient descent
# better than nGPT's version, prevents division by 0 error
def justnorm(x, dim = -1, eps = 1e-12):
    res = x / (x.norm(p=2, dim=dim, keepdim=True) + eps)
    return res

# Cast the input to the correct input layer dtype
def cast_linear(x, layer):
    w = layer.weight.to(x.dtype)
    b = None if layer.bias is None else layer.bias.to(x.dtype)
    return F.linear(x,w,b)


# Hugging Face Config Class (for future deployment)
class HELMConfig(PretrainedConfig):

    model_type = "helm_7c"

    def __init__(
        self,
        # General Model Hyperparameters
        hidden_size = 1024,
        sqrt_hidden_size = 32,
        max_position_embeddings = 4096,
        initializer_range = 0.03125,
        num_hidden_layers = 12,
        num_attention_heads = 32,
        d_head = 64,
        rope_theta = 160000,
        intermediate_size = 2816,
        norm_eps = 1e-12,
        hidden_act = "swiglu",
        swiglu_s_init = 1.0,
        base_lr = 3e-4,
        min_lr = 3e-5,
        weight_decay = 0.0,
        bias = False,
        use_ckpt = False,

        # Tokenization and Data Collator Hyperparameters
        tokenizer_path = "answerdotai/ModernBERT-base",
        vocab_size = 50368,
        bos_token_id = 50281,
        eos_token_id = 50282,
        pad_token_id = 50283,
        mask_token_id = 50284,
        unk_token_id = 50285,
        mlm_probability = 0.3,
        mlm_use_span_masking = True,
        mlm_span_length = 3,

        # HELM_7c Router
        num_router_latents = 4,
        num_permanent_heads = 8,
        head_target_min = 8,
        head_target_center = 16,
        head_target_max = 32,
        easiness_cdf_breakpoints = None,
        count_loss_lambda = 0.5,
        router_grad_clip = 0.05,

        # Permanent-head training noise
        jitter_noise = 0.01,

        # nGPT self attention and FFN hyperparameters
        ngpt_sqk_init_value = 1.0,
        ngpt_sqk_init_scale = 0.03125,
        use_exclusive_attention = True,
        ngpt_alpha_value_attn = 0.05,
        ngpt_alpha_scale_attn = 0.03125,
        ngpt_alpha_value_mlp = 0.05,
        ngpt_alpha_scale_mlp = 0.03125,
        ngpt_suv_value = 1.0,
        ngpt_suv_scale = 1.0,
        ngpt_sz_init_value = 1.00,
        ngpt_sz_init_scale = 0.03125,

        dataset_total_steps = 65000,
        **kwargs
    ):
        # General model
        self.hidden_size = hidden_size
        self.sqrt_hidden_size = sqrt_hidden_size
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.d_head = d_head
        self.rope_theta = rope_theta
        self.intermediate_size = intermediate_size
        self.norm_eps = norm_eps
        self.hidden_act = hidden_act
        self.swiglu_s_init = swiglu_s_init
        self.base_lr = base_lr
        self.min_lr = min_lr
        self.weight_decay = weight_decay
        self.bias = bias
        self.use_ckpt = use_ckpt

        # Tokenization / MLM
        self.tokenizer_path = tokenizer_path
        self.vocab_size = vocab_size
        self.bos_token_id = bos_token_id
        self.eos_token_id = eos_token_id
        self.pad_token_id = pad_token_id
        self.mask_token_id = mask_token_id
        self.unk_token_id = unk_token_id
        self.mlm_probability = mlm_probability
        self.mlm_use_span_masking = mlm_use_span_masking
        self.mlm_span_length = mlm_span_length

        # HELM_7c Router
        self.num_router_latents = num_router_latents
        self.num_permanent_heads = num_permanent_heads
        self.head_target_min = head_target_min
        self.head_target_center = head_target_center
        self.head_target_max = head_target_max
        self.easiness_cdf_breakpoints = easiness_cdf_breakpoints
        self.count_loss_lambda = count_loss_lambda
        self.router_grad_clip = router_grad_clip
        self.jitter_noise = jitter_noise

        elastic = num_attention_heads - num_permanent_heads
        if num_permanent_heads != head_target_min:
            raise ValueError(
                "HELM_7c uses permanent heads as the structural minimum; "
                "num_permanent_heads must equal head_target_min."
            )
        if elastic <= 0:
            raise ValueError("HELM_7c requires at least one elastic head")
        if not (head_target_min <= head_target_center <= head_target_max <= num_attention_heads):
            raise ValueError("Invalid HELM_7c head targets")

        # nGPT
        self.ngpt_sqk_init_value = ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = ngpt_sqk_init_scale
        self.use_exclusive_attention = use_exclusive_attention
        self.ngpt_alpha_value_attn = ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = ngpt_alpha_scale_mlp
        self.ngpt_suv_value = ngpt_suv_value
        self.ngpt_suv_scale = ngpt_suv_scale
        self.ngpt_sz_init_value = ngpt_sz_init_value
        self.ngpt_sz_init_scale = ngpt_sz_init_scale
        self.dataset_total_steps = dataset_total_steps

        super().__init__(**kwargs)


class HELMEmbedding(nn.Module):

    # Initialize Embedding Layer
    def __init__(self, config):
        super().__init__()

        # Embedding Matrix size() : [vocab_size, hidden_size]
        self.word_embeddings = nn.Embedding(
            config.vocab_size,
            config.hidden_size,
            padding_idx=config.pad_token_id
        )

    # Forward Pass (yes, its literally 3 lines)
    def forward(self, input_ids):

        # Map input_ids from Word Embeddings
        word_embeds = self.word_embeddings(input_ids)

        # Normalize (an nGPT must to allow cos. sim. to work)
        embeddings = justnorm(word_embeds)

        # Return
        return embeddings




# HELM_7c multi-latent router
class HELMMultiViewRouter(nn.Module):
    """Minimal sequence-level elastic router for HELM_7c.

    There are 8 permanent heads and 24 elastic candidates. The elastic router uses
    ordinary learned logits z_h(x). Forward routing is hard: z_h > 0. The same hard
    mask is wrapped in a sigmoid STE so CE and the count loss can train the router.

    Easiness labels are training-time supervision only. They are converted to a
    desired total head count in [8, 32]. At inference no easiness value is needed.
    """

    def __init__(self, config):
        super().__init__()
        self.config = config
        self.scale = config.sqrt_hidden_size
        self.num_elastic_candidates = config.num_attention_heads - config.num_permanent_heads

        self.q_down_proj = nn.Linear(
            config.hidden_size,
            config.num_router_latents,
            bias=config.bias,
        )
        self.l_i_weights = nn.Parameter(torch.ones(config.num_router_latents))

        # IMPORTANT: unlike Phase 13, q_up_proj is NOT normalized. Magnitude is allowed
        # to carry information. We monitor its norms instead of pre-emptively constraining it.
        self.q_up_proj = nn.Linear(
            config.hidden_size,
            self.num_elastic_candidates,
            bias=False,
        )

        # Last-forward telemetry.
        self.save_router_logits = None
        self.save_sigmoid_scores = None
        self.save_hard_mask = None
        self.save_total_head_count = None
        self.save_target_total_head_count = None
        self.save_count_error = None
        self.save_count_loss = None

    def _easiness_to_target(self, easiness_score):
        """Map easiness label -> integer target total heads in [8, 32].

        Easiness is converted to a CDF quantile q so the target depends on relative
        difficulty rather than the raw label's dataset-specific numeric scale:
          q=0   (hardest) -> 32 total heads
          q=0.5 (median)  -> 16 total heads
          q=1   (easiest) -> 8 total heads
        """
        batch = easiness_score.numel()
        device = easiness_score.device
        e = easiness_score.to(torch.float32).reshape(batch).clamp(0.0, 1.0)

        bp = self.config.easiness_cdf_breakpoints
        if bp is not None and len(bp) >= 2:
            breaks = torch.as_tensor(bp, device=device, dtype=torch.float32)
            n_intervals = breaks.numel() - 1
            pos = torch.searchsorted(breaks, e, right=True).clamp(1, n_intervals)
            lo = breaks[pos - 1]
            hi = breaks[pos]
            frac = (e - lo) / (hi - lo + 1e-8)
            q = ((pos - 1).to(torch.float32) + frac) / float(n_intervals)
            q = q.clamp(0.0, 1.0)
        else:
            # Safe fallback if no breakpoint table was supplied.
            q = e

        h_min = float(self.config.head_target_min)
        h_ctr = float(self.config.head_target_center)
        h_max = float(self.config.head_target_max)

        hard_half = q < 0.5
        hard_target = h_ctr + (h_max - h_ctr) * ((0.5 - q) / 0.5)
        easy_target = h_ctr + (h_min - h_ctr) * ((q - 0.5) / 0.5)
        target_total = torch.where(hard_half, hard_target, easy_target)

        # Actual executed counts are integer, so make an exactly attainable target.
        return target_total.round().clamp(h_min, h_max)

    def forward(self, hidden_states, easiness_score=None):
        # ----- Existing HELM multi-latent sequence summary -----
        q_down = justnorm(self.q_down_proj.weight, dim=1).to(hidden_states.dtype)
        scanner = F.linear(hidden_states, q_down)                       # [B,S,R]
        scanner_weights = F.softmax(self.scale * scanner, dim=1)       # [B,S,R]
        latents = torch.bmm(scanner_weights.transpose(1, 2), hidden_states)  # [B,R,D]

        latent_weights = F.softmax(self.l_i_weights, dim=0)
        pooled = (latents * latent_weights.view(1, -1, 1)).sum(dim=1)  # [B,D]

        # ----- Minimal learned router -----
        router_logits = cast_linear(pooled, self.q_up_proj)                 # [B,E]
        sigmoid_scores = torch.sigmoid(router_logits)
        hard_mask = (router_logits > 0).to(router_logits.dtype)

        # Forward = exact 0/1 hard mask. Backward = sigmoid derivative.
        ste_mask = hard_mask.detach() - sigmoid_scores.detach() + sigmoid_scores

        actual_elastic_count = hard_mask.sum(dim=-1)
        actual_total_count = actual_elastic_count + float(self.config.num_permanent_heads)

        # ----- Easiness-supervised ACTUAL hard-count loss -----
        if easiness_score is not None:
            target_total_count = self._easiness_to_target(easiness_score)
            target_elastic_count = target_total_count - float(self.config.num_permanent_heads)

            # ste_mask has the hard count as its forward value but keeps a sigmoid
            # backward path. This avoids the old sum(sigmoid) soft-count loophole.
            differentiable_elastic_count = ste_mask.float().sum(dim=-1)
            count_error = differentiable_elastic_count - target_elastic_count.float()
            denom = float(self.num_elastic_candidates)
            count_loss = (
                float(self.config.count_loss_lambda)
                * (count_error / denom).square().mean()
            )
        else:
            if self.training:
                raise ValueError("HELM_7c training requires easiness_score")
            target_total_count = torch.full_like(actual_total_count, -1.0)
            count_error = torch.zeros_like(actual_total_count)
            count_loss = router_logits.new_zeros(())

        # ----- Telemetry -----
        self.save_router_logits = router_logits.detach()
        self.save_sigmoid_scores = sigmoid_scores.detach()
        self.save_hard_mask = hard_mask.detach()
        self.save_total_head_count = actual_total_count.detach()
        self.save_target_total_head_count = target_total_count.detach()
        self.save_count_error = (actual_total_count - target_total_count).detach()
        self.count_loss = count_loss
        self.save_count_loss = count_loss.detach()

        router_mask = ste_mask.view(ste_mask.size(0), -1, 1, 1)
        if self.config.num_permanent_heads > 0:
            permanent = torch.ones(
                ste_mask.size(0),
                self.config.num_permanent_heads,
                1,
                1,
                device=router_mask.device,
                dtype=router_mask.dtype,
            )
            router_mask = torch.cat((permanent, router_mask), dim=1)

        return router_mask


class RotaryEmbeddings(nn.Module):

    # Initialize the Following
    # rope_theta
    # max_position_embeddings
    # sin & cos table
    def __init__(self, dim, max_position_embeddings, rope_theta = 160000):
        super().__init__()

        # Define inverse of frequencies
        # size(): [dim/2]
        inv_freq = 1.0 / (rope_theta ** (torch.arange(0, dim, 2).float() / dim))

        # Create position vector
        # size(): [max_position_embeddings]
        t = torch.arange(max_position_embeddings, dtype = inv_freq.dtype)

        freqs = torch.outer(t, inv_freq)

        freqs = torch.cat((freqs, freqs), dim = -1)


        # Save the Sine and Cosine
        self.register_buffer("cos", freqs.cos())
        self.register_buffer("sin", freqs.sin())

    # Implement rotate_half (Allows for clean rotation mechanics)
    def rotate_half(self, x):

        # Take x as the first half
        x1 = x[..., : x.shape[-1] // 2]

        # Take y was the second half
        x2 = x[..., x.shape[-1] // 2 :]

        return torch.cat((-x2, x1), dim = -1)


    # Implement apply_rotary_embeddings
    # Does RoPE
    # Expected input size: [b, num_attention_heads, seq_len, dim]
    # Output: [b, num_attention_heads, seq_len, dim]
    def forward(self, x):

        # Get token length
        seq_len = x.shape[-2]

        # Take a slice of the cos and sin tables
        x_cos = self.cos[:seq_len, ...].to(dtype=x.dtype)
        x_sin = self.sin[:seq_len, ...].to(dtype=x.dtype)

        # Return RoPE matrix
        return (x * x_cos) + (self.rotate_half(x) * x_sin)



# Self Attention
# Literally Just Self Attention
# QKV cross self attention
# Use RoPE
# Output Matrix
# Speicfics about training (masked training)
# MODIFICATION: USE FLEX ATTENTION TO ALLOW FOR BATCHED INFERENCE
class HELMSelfAttention(nn.Module):

    # Initialize the following:
    #   - QKV matrix
    #   - Output matrix
    #   - Scaling vector sqk for q and k
    #   - RoPE Module
    def __init__(self, config):
        super().__init__()

        # Grabbing config values from convience
        self.hidden_size = config.hidden_size
        self.num_attention_heads = config.num_attention_heads
        self.num_permanent_heads = config.num_permanent_heads
        self.d_head = config.d_head if config.d_head is not None else (config.hidden_size // config.num_attention_heads)
        self.total_head_dim = self.num_attention_heads * self.d_head   
        self.ngpt_sqk_init_value = config.ngpt_sqk_init_value
        self.ngpt_sqk_init_scale = config.ngpt_sqk_init_scale
        self.config = config

        self._eval_backend = "dense"
        self._flex_compiled = False
        self._flex_fn = None
        self._block_mask_fn = None


        # QKV Matrix
        self.qkv = nn.Linear(
            config.hidden_size,
            self.total_head_dim * 3,
            bias = config.bias
        )

        # RoPE Module
        self.RoPE = RotaryEmbeddings(
            self.d_head,
            config.max_position_embeddings,
            config.rope_theta
        )

        # SQK scalers right after RoPE
        self.sqk = nn.Parameter(self.ngpt_sqk_init_scale*torch.ones(self.total_head_dim))  # was: self.hidden_size

        # Output Matrix
        self.output = nn.Linear(
            self.total_head_dim,      # was: config.hidden_size
            config.hidden_size,
            bias = config.bias
        )

    # Configure the eval-time attention backend. Call via model.enable_efficient_inference(...).
    #   backend="flex"  : FlexAttention; set compile=True on GPU for the fused kernel (recommended).
    #   backend="gather": compact gather/scatter SDPA, no torch.compile needed.
    #   backend="dense" : compute-all-then-mask (default; what training uses).
    def set_eval_backend(self, backend="flex", compile=True):
        compile = bool(compile)
        # Only drop the cached torch.compile()'d function/block-mask builder when the
        # backend or compile flag actually changes -- resetting on every call (even when
        # nothing changed) forces a full recompilation on the very next forward pass,
        # which is silently expensive if this is called before every timed benchmark run.
        changed = (backend != getattr(self, "_eval_backend", None)
                   or compile != getattr(self, "_flex_compiled", None))
        self._eval_backend = backend
        self._flex_compiled = compile
        if changed:
            self._flex_fn = None
            self._block_mask_fn = None

    def _flex_attn(self, q, k, v, block_mask, scale):
        if self._flex_fn is None:
            from torch.nn.attention.flex_attention import flex_attention
            self._flex_fn = torch.compile(flex_attention) if self._flex_compiled else flex_attention
        return self._flex_fn(q, k, v, block_mask=block_mask, scale=scale)

    def _build_block_mask(self, mask_mod, B, H, S, device):
        if self._block_mask_fn is None:
            from torch.nn.attention.flex_attention import create_block_mask
            # compiling create_block_mask avoids materializing the full SxS mask for long sequences
            self._block_mask_fn = torch.compile(create_block_mask) if self._flex_compiled else create_block_mask
        return self._block_mask_fn(mask_mod, B, H, S, S, device=device)

    # Define Training
    def forward(self, hidden_states, attention_mask, router_mask):

        # Obtain projection from hidden_states onto QKV
        # size(): [b, seq_len, hidden_size * 3]
        qkv_proj = cast_linear(hidden_states, self.qkv)

        # Obtain Hidden Size
        batch_size, seq_len, _ = hidden_states.size()

        # Split Projects
        # q, k, v size(): [b, seq_len, hidden_size]
        q, k, v = qkv_proj.split(self.total_head_dim, dim=-1)

        # Define sqk for scaling q, k, and v
        # size(): [hidden_size]
        sqk = (self.sqk * (self.ngpt_sqk_init_value/self.ngpt_sqk_init_scale))
        # Resizing is required for when we element-wise multiply this by q and k matrice:s [1, num_attention_heads, 1, d_head] * [b, num_attention_heads, seq_len, hidden_size]
        # size(): [hidden_size]-> [1, num_attention_heads, 1, d_head]
        sqk = sqk.view(1, self.num_attention_heads, 1, self.d_head)


        eval_backend = self._eval_backend

        # Reshape q,k,v
        # q, k, v size(): [b, seq_len, num_attention_heads, d_head]
        q = q.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        k = k.view(batch_size, seq_len, self.num_attention_heads, self.d_head)
        v = v.view(batch_size, seq_len, self.num_attention_heads, self.d_head)

        # Reshape q,k,v
        # q, k, v size(): [b, num_attention_heads, seq_len, d_head]
        q = q.permute(0,2,1,3)
        k = k.permute(0,2,1,3)
        v = v.permute(0,2,1,3)


        # TRAINING / TPU MODE
        if (self.training or eval_backend == "dense"):

            # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Apply Attention
            # Scale by sqrt(dk)
            # A whole lot happens here. final size(): [b, num_attention_heads, seq_len, d_head]
            context_layer = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head),
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            if router_mask is not None:
                # Apply Broadcasting Mask (expand_as() good for XLA)
                # size(): [b, num_attention_heads, seq_len, d_head]
                context_layer = context_layer * router_mask.expand_as(context_layer)

            # Apply Jitter Noise to the Permanent heads during training
            if self.training and self.num_permanent_heads > 0:

                # Take the permanent heads:
                permanent_heads = context_layer[:,:self.num_permanent_heads, :, :]

                # Take the elastic heads:
                elastic_heads = context_layer[:, self.num_permanent_heads:, :, :]

                # Apply dropout
                permanent_heads = F.dropout(permanent_heads, p = self.config.jitter_noise, training = self.training)

                # Combine back together
                context_layer = torch.cat((permanent_heads, elastic_heads),dim = 1)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # FLEX ATTENTION (for GPUs)
        elif eval_backend == "flex" and batch_size > 1:

             # Normalize q and k
            q = justnorm(q)
            k = justnorm(k)

            # Apply RoPE
            q = self.RoPE(q)
            k = self.RoPE(k)

            # Apply sqk scaling factor to q and k
            q = sqk.to(q.dtype) * q
            k = sqk.to(k.dtype) * k

            # Router_mask scores, 1 or 0 or sigmoid scaling
            # [b, num_attention_heads, 1 , 1] -> [batch, num_attention_heads]
            active = (router_mask[:, :, 0, 0] > 0)

            # Boolean attention mask
            # [batch_size, 1, 1, seq_len] -> [batch, seq_len]
            key_valid = (attention_mask[:, 0, 0, :] >=0)

            # mask_mod: attend / calculate only if the head is on and its not a padding token
            def mask_mod(bi, hi, qi, ki):
                return active[bi, hi] & key_valid[bi, ki]

            # Prep the block to be passed into flex attention
            block_mask = self._build_block_mask(
                mask_mod, batch_size, self.num_attention_heads,seq_len, q.device
            )

            # Apply flex attention
            context_layer = self._flex_attn(
                q, k, v, block_mask = block_mask, scale = math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v, dim=-1)
                context_layer = context_layer - (context_layer * Vn).sum(dim=-1, keepdim=True) * Vn

            # Apply router mask to 0 the heads of the context layer
            # [batch, num attention heads, seq_len, head dim] (router_mask [batch, num_attention_heads, 1,1] was broadcasted)
            context_layer = context_layer * router_mask.expand_as(context_layer)

            # Reshape
            # size(): [b, seq_len, num_attention_heads, d_head]
            context_reshaped = context_layer.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions:
            # size(): [b, seq_len, num_hidden_size]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # Project context onto the Output Matrix
            context_layer = cast_linear(context_reshaped, self.output)

        # Single query effieincy
        else:

            # This path only looks at batch element 0's router decisions (see below), so
            # it is only correct for batch_size == 1 -- each example's active heads are
            # data-dependent, so silently reusing example 0's mask for other examples
            # would produce wrong outputs for them instead of a loud failure.
            assert batch_size == 1, (
                f"HELMSelfAttention's 'gather' eval backend only supports batch_size == 1 "
                f"(got batch_size={batch_size}); use backend='flex' for batched inference."
            )

            # Find the heads that are on
            # nonzero(): [1, num_attention_heads, 1, 1] -> [num_active_heads, 1]
            # squeeze(): [num_active_heads, 1] -> [num_active_heads] (indices)
            active_indices = torch.nonzero(router_mask[0, :, 0, 0]).squeeze(-1)

            # q, k, v are already [b, num_attention_heads, seq_len, d_head] from the
            # shared reshape/permute above -- no need to reshape them again here.

            # 2. Extract the parts used by the active heads
            # size(): [1, num_attention_heads, seq_len, d_head] ->  [1, num_active_heads, seq_len, d_head]
            q_sliced = q[:, active_indices, :, :]
            k_sliced = k[:, active_indices, :, :]
            v_sliced = v[:, active_indices, :, :]

            # Normalize q and k
            q_sliced = justnorm(q_sliced)
            k_sliced = justnorm(k_sliced)

            # Apply RoPE
            q_sliced = self.RoPE(q_sliced)
            k_sliced = self.RoPE(k_sliced)

            # Apply sqk scaling factor to q and k
            sqk_sliced = sqk[:, active_indices, :, :]
            q_sliced = sqk_sliced.to(q_sliced.dtype) * q_sliced
            k_sliced = sqk_sliced.to(k_sliced.dtype) * k_sliced

            # Flash Attention (only for GPUs where on-the-fly splicing can exist)
            # size(): [b, num_active_heads, seq_len, d_head]
            context_sliced = F.scaled_dot_product_attention(
                q_sliced, k_sliced, v_sliced,
                attn_mask=attention_mask.to(q.dtype),
                scale=math.sqrt(self.d_head)
            )

            # Add Exclusive Attention (better results?)
            if (self.config.use_exclusive_attention):
                Vn = torch.nn.functional.normalize(v_sliced, dim=-1)
                context_sliced = context_sliced - (context_sliced * Vn).sum(dim=-1, keepdim=True) * Vn

            # STE tie to the router
            # Note: If use_sigmoid_scaling = True: Scales the router mask back to the sigmoid values
            # (since active indices were just indices of the values, not the real values)
            # If use_sigmooid_scaling = False, then multiplying by 1 does mathimatically nothing
            active_weights = router_mask[:, active_indices, :, :]
            context_sliced = context_sliced * active_weights

            # 5. Reshape for the output linear layer
            # [1, num_active, seq_len, d_head] -> [1, seq_len, num_active, d_head]
            context_reshaped = context_sliced.permute(0, 2, 1, 3).contiguous()

            # Flatten the last two dimensions: [1, seq_len, num_active * d_head]
            context_reshaped = context_reshaped.view(batch_size, seq_len, -1)

            # 6. Map the active head indices to their exact hidden dimension indices
            # Example: Head 1 with d_head=64 generates indices 64 through 127
            dim_offsets = torch.arange(self.d_head, device=hidden_states.device)
            active_dims = (active_indices.unsqueeze(1) * self.d_head + dim_offsets).view(-1)

            # 7. Slice the input columns of the output weight matrix
            # original shape [hidden_size, hidden_size] -> [hidden_size, num_active * d_head]
            sliced_weight = self.output.weight[:, active_dims].to(context_reshaped.dtype)
            sliced_bias = None if self.output.bias is None else self.output.bias.to(context_reshaped.dtype)

            # 8. Perform the compressed functional linear projection
            context_layer = F.linear(context_reshaped, sliced_weight, bias=sliced_bias)

        # Return context_layer (normalization occurs in HELMMLP)
        return context_layer



# HELMMLP (FFN of nGPT architecture)
# All of this stays the same from the original nGPT paper
class HELMMLP(nn.Module):

    # Define the Following:
    #   - Constants from config (for convience?)
    #       * hidden_size
    #       * ngpt_alpha_value_attn
    #       * ngpt_alpha_scale_attn
    #       * ngpt_alpha_value_mlp
    #       * ngpt_alpha_scale_mlp
    #       * ngpt_suv_value
    #       * ngpt_suv_scale
    #   - Eigen learning rate after attention (attn_alpha)
    #   - Eigen learning rate after mlp (mlp_alpha)
    #   - MLP expansion layer (mlp_exp)
    #   - suv scaling vectors for SwiGLU (suv)
    #   - SiLU() activation (silu)
    #   - MLP projection layer (mlp_expand)
    def __init__(self, config):
        super().__init__()

        # Gather Config Values for convience
        self.hidden_size = config.hidden_size
        self.ngpt_alpha_value_attn = config.ngpt_alpha_value_attn
        self.ngpt_alpha_scale_attn = config.ngpt_alpha_scale_attn
        self.ngpt_alpha_value_mlp = config.ngpt_alpha_value_mlp
        self.ngpt_alpha_scale_mlp = config.ngpt_alpha_scale_mlp
        self.ngpt_suv_value = config.ngpt_suv_value
        self.ngpt_suv_scale = config.ngpt_suv_scale
        self.intermediate_size = config.intermediate_size

        # Alpha Eigen Update after Attention (1st Optimizer Step)
        self.attn_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_attn*torch.ones(self.hidden_size))

        # Alpha Eigen Update after MLP (2nd Optimizer Step)
        self.mlp_alpha = torch.nn.Parameter(self.ngpt_alpha_scale_mlp*torch.ones(self.hidden_size))

        # MLP expansion layer
        self.mlp_exp = nn.Linear(
            self.hidden_size,
            2 * self.intermediate_size,
            bias = config.bias
        )

        # suv scaling vectors during SwiGLU
        self.suv = torch.nn.Parameter(self.ngpt_suv_scale*torch.ones(2 * self.intermediate_size))

        # Define SiLU()
        self.silu = nn.SiLU()

        # MLP projection layer (shrink)
        self.mlp_proj  = nn.Linear(
            self.intermediate_size,
            self.hidden_size,
            bias=config.bias
        )

    # Peform MLP from the output of the output matrix to the end of the transformer block
    def forward(self, hidden_states, hidden_states_attention):

        # Even more convience
        hidden_size = self.hidden_size
        ngpt_alpha_value_attn = self.ngpt_alpha_value_attn
        ngpt_alpha_scale_attn = self.ngpt_alpha_scale_attn
        ngpt_alpha_value_mlp = self.ngpt_alpha_value_mlp
        ngpt_alpha_scale_mlp = self.ngpt_alpha_scale_mlp
        ngpt_suv_value = self.ngpt_suv_value
        ngpt_suv_scale = self.ngpt_suv_scale

        # Mostly Lifted from the nGPT model.py

        # Apply Normalization to hidden states before and after attention
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states)
        B_norm = justnorm(hidden_states_attention)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.attn_alpha * (ngpt_alpha_value_attn / ngpt_alpha_scale_attn)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_a * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt1 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt1 = justnorm(hidden_states_opt1)

        # Get u and v matrices by multiplying by mlp_exp
        # size(): [b, seq_len, hidden_size] * [hidden_size, 2 * intermediate_size] = [b, seq_len, 2 * intermediate_size]
        uv_pre = cast_linear(hidden_states_opt1 ,self.mlp_exp)
        # prepare scaling vector suv
        # size(): [intermediate_size * 2] (remember, they are concatenated)
        suv = self.suv * (ngpt_suv_value/ngpt_suv_scale) * (hidden_size ** 0.5)
        # We need to keep suv to be bf16. The line above promoted suc fp32 and the autocaster didn't fix it
        suv = suv.to(uv_pre.dtype)

        # element-wise uv by scaling vector suv
        # size(): [b, seq_len, 2 * intermediate_size]
        uv_post_suv = suv * uv_pre

        # Chunk uv into u and v
        # both size(): [b, seq_len, intermediate_size]
        u, v = torch.chunk(uv_post_suv, 2, dim=-1)

        # Apply u * silu(v), the whole point of SwiGLU (element-wise)
        # size(): [b, seq_len, intermediate_size]
        x_mlp = u * self.silu(v)

        # Project x_mlp to the mlp_proj layer (shrink)
        # size(): [b, seq_len, intermediate_size] * [intermediate_size, hidden_size] = [b, seq_len, hidden_size]
        h_mlp = cast_linear(x_mlp, self.mlp_proj)

        # Apply Normalization to hidden states after attention and after mlp
        # both size(): [b, seq_len, hidden_size]
        A_norm = justnorm(hidden_states_opt1)
        B_norm = justnorm(h_mlp)

        # Define the eigen learning rate
        # alpha >=0
        # size(): [hidden_size]
        lr = self.mlp_alpha * (ngpt_alpha_value_mlp / ngpt_alpha_scale_mlp)
        lr = torch.abs(lr).to(A_norm.dtype)

        # h = Norm(h + alpha_m * (h_a - h)) (element-wise)
        # size(): [b, seq_len, hidden_size]
        hidden_states_opt2 = A_norm + lr * (B_norm - A_norm)
        hidden_states_opt2 = justnorm(hidden_states_opt2)

        # Return new hidden_state
        return hidden_states_opt2



# HELMBLOCK = HELMMultiViewRouter + HELMSelfAttention + HELMMLP
class HELMBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.mlt_vw_rtr = HELMMultiViewRouter(config)
        self.attn = HELMSelfAttention(config)
        self.mlp = HELMMLP(config)

    def forward(self, hidden_states, attention_mask, easiness_score):
        router_mask = self.mlt_vw_rtr(hidden_states, easiness_score)
        count_loss = self.mlt_vw_rtr.count_loss
        attn_output = self.attn(hidden_states, attention_mask, router_mask)
        layer_output = self.mlp(hidden_states, attn_output)
        return layer_output, count_loss


class HELMModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.use_ckpt = config.use_ckpt
        self.embedding = HELMEmbedding(config)
        self.blocks = nn.ModuleList([HELMBlock(config) for _ in range(config.num_hidden_layers)])

    def forward(self, input_ids, attention_mask, easiness_score=None):
        attention_mask = attention_mask.unsqueeze(1).unsqueeze(2).to(torch.bfloat16)
        attention_mask = attention_mask.masked_fill(attention_mask == 0, float('-inf'))
        attention_mask = attention_mask.masked_fill(attention_mask == 1, 0.0)

        hidden_states = self.embedding(input_ids).to(torch.bfloat16)
        total_count_loss = hidden_states.new_zeros(())

        for block in self.blocks:
            if self.use_ckpt and self.training:
                _ckpt = (_xla_checkpoint if (_xla_checkpoint is not None
                         and hidden_states.device.type == "xla")
                         else torch.utils.checkpoint.checkpoint)
                hidden_states, count_loss = _ckpt(
                    block,
                    hidden_states,
                    attention_mask,
                    easiness_score,
                    use_reentrant=True if hidden_states.device.type == "xla" else False,
                )
            else:
                hidden_states, count_loss = block(hidden_states, attention_mask, easiness_score)
            total_count_loss = total_count_loss + count_loss

        # Count supervision is per-layer; average so lambda is independent of depth.
        total_count_loss = total_count_loss / float(len(self.blocks))
        return hidden_states, total_count_loss


class HELMForMaskedLM(PreTrainedModel):

    config_class = HELMConfig

    def __init__(self, config):
        super().__init__(config)
        self.ngpt_sz_init_value = config.ngpt_sz_init_value
        self.ngpt_sz_init_scale = config.ngpt_sz_init_scale
        self.model = HELMModel(config)
        self.classifier = nn.Linear(config.hidden_size, config.vocab_size, bias=config.bias)
        self.sz = nn.Parameter(torch.ones(config.vocab_size))
        self.post_init()

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)

    def enable_efficient_inference(self, backend="flex", compile=True):
        for block in self.model.blocks:
            block.attn.set_eval_backend(backend=backend, compile=compile)
        return self

    @torch.no_grad()
    def normalize_ngpt_matrices(self):
        # q_up_proj is intentionally EXCLUDED: HELM_7c allows router magnitude.
        keys_to_normalize = (
            "word_embeddings.weight",
            "classifier.weight",
            "attn.qkv.weight",
            "attn.output.weight",
            "mlp.mlp_exp.weight",
            "mlp.mlp_proj.weight",
            "mlt_vw_rtr.q_down_proj.weight",
        )
        for name, param in self.named_parameters():
            if name.endswith(keys_to_normalize):
                param.copy_(justnorm(param, dim=1, eps=1e-12))

    @torch.no_grad()
    def get_telemetry(self):
        telemetry = {}

        for i, block in enumerate(self.model.blocks):
            router = block.mlt_vw_rtr
            logits = router.save_router_logits.float().cpu()
            sigmoid = router.save_sigmoid_scores.float().cpu()
            hard = router.save_hard_mask.float().cpu()
            actual = router.save_total_head_count.float().cpu()
            target = router.save_target_total_head_count.float().cpu()
            error = router.save_count_error.float().cpu()
            q_up_norms = router.q_up_proj.weight.detach().float().norm(dim=1).cpu()

            telemetry[f"layer_{i}_router_logits"] = logits
            telemetry[f"layer_{i}_sigmoid_scores"] = sigmoid
            telemetry[f"layer_{i}_hard_mask"] = hard
            telemetry[f"layer_{i}_elastic_head_ratio"] = hard.mean().item()
            telemetry[f"layer_{i}_total_head_count_mean"] = actual.mean().item()
            telemetry[f"layer_{i}_target_head_count_mean"] = target.mean().item()
            telemetry[f"layer_{i}_count_error_mean"] = error.mean().item()
            telemetry[f"layer_{i}_count_error_mae"] = error.abs().mean().item()
            telemetry[f"layer_{i}_count_loss"] = router.save_count_loss.float().item()
            telemetry[f"layer_{i}_router_weight_norms"] = q_up_norms
            telemetry[f"layer_{i}_router_weight_norm_mean"] = q_up_norms.mean().item()
            telemetry[f"layer_{i}_router_weight_norm_std"] = q_up_norms.std(unbiased=False).item()
            telemetry[f"layer_{i}_l_i_weights"] = router.l_i_weights.detach().float().cpu()

        return telemetry

    def forward(self, input_ids, attention_mask, current_step=None, easiness_score=None):
        # current_step is accepted only for backward compatibility with older callers.
        features, total_count_loss = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            easiness_score=easiness_score,
        )

        sz = self.sz * (self.ngpt_sz_init_value / self.ngpt_sz_init_scale)
        unscaled_logits = cast_linear(features, self.classifier)
        logits = sz.to(unscaled_logits.dtype) * unscaled_logits
        return logits, total_count_loss



Writing model.py


In [6]:
%%writefile analyze_helm7c_heads.py
"""
HELM_7c checkpoint head-dynamics / redundancy diagnostic.

Default checkpoint:
  repo: JamesResearch1216/HELM_7c
  file: checkpoint-006500.pt

What this script tests
----------------------
1) Router/cardinality behavior
   - actual head counts vs easiness-derived targets
   - easiness/count Spearman correlation
   - per-head activation frequency and mask diversity
   - sigmoid saturation and STE derivative health
   - router logit variance across inputs vs across heads

2) Quality of routing decisions
   - routed CE
   - forced-dense CE (all 32 heads)
   - permanent-only CE (8 permanent heads)
   - random-same-count CE (same compute as HELM, random elastic identities)
   - first-order same-count "oracle" CE on a small subset

3) Router/utility alignment
   - freeze all model parameters
   - temporarily replace each layer's router mask by differentiable per-example gates
   - force all heads on and compute |d CE / d gate_h|
   - compare this CE importance to HELM's elastic router logits
   - measure active-vs-inactive importance and top-k overlap

4) Functional head redundancy
   - capture the hidden state entering layers 0, 5, and 11 (configurable)
   - recompute ALL 32 heads' attention context BEFORE router masking
   - project every head separately through its own slice of W_O
   - compare residual-stream contributions Y_h = C_h W_O^(h)
   - report pairwise cosine similarity and effective rank

5) Parameter redundancy (secondary diagnostic)
   - pairwise cosine similarity of concatenated Q/K/V/O head parameter blocks

The script intentionally uses a deterministic diagnostic MLM mask. Its CE values are
for RELATIVE comparisons inside this script, not for reproducing the trainer's exact
validation loss.

Outputs
-------
<output_dir>/
  summary.md
  summary.json
  count_calibration.csv
  layer_XX_head_stats.csv
  layer_XX_functional_similarity.csv
  layer_XX_parameter_similarity.csv
  layer_XX_*.png
  utility_alignment.csv
  HELM_7c_head_analysis_results.zip

Run examples
------------
GPU:
  python analyze_helm7c_heads.py --device cuda

Single TPU/XLA device (after your normal Kaggle TPU repair/restart cell):
  python analyze_helm7c_heads.py --device xla --batch-size 2

CPU (slow):
  python analyze_helm7c_heads.py --device cpu --num-examples 8 --seq-len 256

For a heavier exact single-head ablation diagnostic:
  python analyze_helm7c_heads.py --device cuda --exact-ablation
"""

from __future__ import annotations

import argparse
import contextlib
import csv
import glob
import json
import math
import os
import random
import shutil
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import torch
import torch.nn.functional as F

# Matplotlib only; no seaborn dependency.
import matplotlib.pyplot as plt

try:
    from huggingface_hub import hf_hub_download
except Exception as exc:
    raise RuntimeError(
        "huggingface_hub is required. Install it with: pip install huggingface_hub"
    ) from exc

try:
    import pyarrow.parquet as pq
except Exception as exc:
    raise RuntimeError("pyarrow is required. Install it with: pip install pyarrow") from exc

# model.py must be next to this script (included in the bundle I provided).
from pathlib import Path
import sys

# Replace line 109 with this:
SCRIPT_DIR = Path.cwd() 

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))


try:
    from model import HELMConfig, HELMForMaskedLM, justnorm, cast_linear
except Exception as exc:
    raise RuntimeError(
        "Could not import HELM_7c model.py. Put the HELM_7c model.py in the same directory "
        "as analyze_helm7c_heads.py."
    ) from exc


# -----------------------------------------------------------------------------
# Constants / defaults
# -----------------------------------------------------------------------------
MODEL_REPO = "JamesResearch1216/HELM_7c"
CHECKPOINT_FILE = "checkpoint-006500.pt"
TRAINING_STATE_FILE = "training_state.json"
DATA_REPO = "JamesResearch1216/HELM-Easiness-Data-10B-Labeled-v6"
VALIDATION_FILE = "data/seq_1024/validation-00000.parquet"


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def get_hf_token() -> Optional[str]:
    """Resolve HF token without requiring one for public repositories."""
    token = os.getenv("HF_TOKEN") or os.getenv("HUGGING_FACE_HUB_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        return None


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def strip_state_prefixes(state: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
    """Handle DDP / torch.compile prefixes defensively."""
    out = {}
    for key, value in state.items():
        k = key
        changed = True
        while changed:
            changed = False
            for prefix in ("module.", "_orig_mod."):
                if k.startswith(prefix):
                    k = k[len(prefix):]
                    changed = True
        out[k] = value
    return out


def double_argsort_rank(x: torch.Tensor, descending: bool = True) -> torch.Tensor:
    """Return integer rank per row, rank 0 = largest when descending=True."""
    order = torch.argsort(x, dim=-1, descending=descending)
    return torch.argsort(order, dim=-1)


def rankdata_np(x: np.ndarray) -> np.ndarray:
    """Simple average-tie rank data, sufficient for diagnostics."""
    x = np.asarray(x)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=np.float64)
    sorted_x = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sorted_x[j] == sorted_x[i]:
            j += 1
        avg_rank = 0.5 * (i + j - 1)
        ranks[order[i:j]] = avg_rank
        i = j
    return ranks


def spearman_np(x: Sequence[float], y: Sequence[float]) -> float:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    y = np.asarray(y, dtype=np.float64).reshape(-1)
    good = np.isfinite(x) & np.isfinite(y)
    x = x[good]
    y = y[good]
    if len(x) < 3 or np.std(x) == 0 or np.std(y) == 0:
        return float("nan")
    rx = rankdata_np(x)
    ry = rankdata_np(y)
    return float(np.corrcoef(rx, ry)[0, 1])


def effective_rank_from_gram(gram: np.ndarray) -> Tuple[float, float, np.ndarray]:
    """Return entropy effective rank, participation rank, eigenvalues."""
    g = 0.5 * (gram + gram.T)
    vals = np.linalg.eigvalsh(g)
    vals = np.clip(vals, 0.0, None)
    total = vals.sum()
    if total <= 1e-12:
        return 0.0, 0.0, vals
    p = vals / total
    p_pos = p[p > 1e-12]
    entropy_rank = float(np.exp(-(p_pos * np.log(p_pos)).sum()))
    participation = float((total * total) / (np.square(vals).sum() + 1e-12))
    return entropy_rank, participation, vals


def cosine_from_gram(gram: np.ndarray) -> np.ndarray:
    diag = np.clip(np.diag(gram), 1e-12, None)
    denom = np.sqrt(np.outer(diag, diag))
    cos = gram / denom
    return np.clip(cos, -1.0, 1.0)


def offdiag_stats(matrix: np.ndarray, threshold: float = 0.80) -> Dict[str, float]:
    n = matrix.shape[0]
    mask = ~np.eye(n, dtype=bool)
    vals = np.abs(matrix[mask])
    if vals.size == 0:
        return {"mean_abs": 0.0, "median_abs": 0.0, "frac_abs_gt_0.8": 0.0, "max_abs": 0.0}
    return {
        "mean_abs": float(vals.mean()),
        "median_abs": float(np.median(vals)),
        "frac_abs_gt_0.8": float((vals > threshold).mean()),
        "max_abs": float(vals.max()),
    }


def top_pairs(matrix: np.ndarray, k: int = 8) -> List[Tuple[int, int, float]]:
    pairs = []
    for i in range(matrix.shape[0]):
        for j in range(i + 1, matrix.shape[1]):
            pairs.append((i, j, float(matrix[i, j])))
    pairs.sort(key=lambda t: abs(t[2]), reverse=True)
    return pairs[:k]


def save_matrix_csv(path: Path, matrix: np.ndarray, prefix: str = "head") -> None:
    with path.open("w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([""] + [f"{prefix}_{i}" for i in range(matrix.shape[1])])
        for i, row in enumerate(matrix):
            writer.writerow([f"{prefix}_{i}"] + [float(v) for v in row])


def save_heatmap(path: Path, matrix: np.ndarray, title: str, xlabel: str, ylabel: str,
                 vmin: Optional[float] = None, vmax: Optional[float] = None) -> None:
    fig, ax = plt.subplots(figsize=(10, 7))
    im = ax.imshow(matrix, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


def save_bar(path: Path, values: np.ndarray, title: str, ylabel: str) -> None:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(np.arange(len(values)), values)
    ax.set_title(title)
    ax.set_xlabel("Head")
    ax.set_ylabel(ylabel)
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


# -----------------------------------------------------------------------------
# Device abstraction
# -----------------------------------------------------------------------------
@dataclass
class DeviceContext:
    device: torch.device
    kind: str
    xm: object = None
    autocast_dtype: Optional[torch.dtype] = None

    def autocast(self):
        if self.kind == "cuda":
            return torch.autocast(device_type="cuda", dtype=self.autocast_dtype)
        if self.kind == "xla":
            try:
                return torch.autocast(device_type="xla", dtype=torch.bfloat16)
            except Exception:
                return contextlib.nullcontext()
        return contextlib.nullcontext()

    def mark_step(self):
        if self.kind == "xla" and self.xm is not None:
            self.xm.mark_step()


def resolve_device(requested: str) -> DeviceContext:
    requested = requested.lower()
    if requested == "auto":
        if torch.cuda.is_available():
            requested = "cuda"
        elif glob.glob("/dev/accel*"):
            requested = "xla"
        else:
            requested = "cpu"

    if requested == "cuda":
        if not torch.cuda.is_available():
            raise RuntimeError("--device cuda requested but CUDA is not available")
        if torch.cuda.is_bf16_supported():
            dtype = torch.bfloat16
        else:
            dtype = torch.float16
        return DeviceContext(torch.device("cuda"), "cuda", autocast_dtype=dtype)

    if requested == "xla":
        # Assumes the user already ran/restarted with the same TPU environment setup used
        # by HELM training.
        try:
            import torch_xla.core.xla_model as xm
        except Exception as exc:
            raise RuntimeError(
                "--device xla requested but torch_xla could not be imported. Run your normal "
                "TPU repair/restart cell first."
            ) from exc
        return DeviceContext(xm.xla_device(), "xla", xm=xm, autocast_dtype=torch.bfloat16)

    if requested == "cpu":
        return DeviceContext(torch.device("cpu"), "cpu", autocast_dtype=None)

    raise ValueError(f"Unknown device: {requested}")


# -----------------------------------------------------------------------------
# Checkpoint / data loading
# -----------------------------------------------------------------------------
def download_assets(cache_dir: Path, token: Optional[str], model_repo: str,
                    checkpoint_file: str, data_repo: str, validation_file: str):
    cache_dir.mkdir(parents=True, exist_ok=True)
    print(f"Downloading checkpoint: {model_repo}/{checkpoint_file}")
    ckpt_path = hf_hub_download(
        repo_id=model_repo,
        filename=checkpoint_file,
        repo_type="model",
        token=token,
        local_dir=str(cache_dir / "model_repo"),
    )

    training_state_path = None
    try:
        training_state_path = hf_hub_download(
            repo_id=model_repo,
            filename=TRAINING_STATE_FILE,
            repo_type="model",
            token=token,
            local_dir=str(cache_dir / "model_repo"),
        )
    except Exception as exc:
        print(f"WARNING: could not download training_state.json: {exc}")

    print(f"Downloading validation shard: {data_repo}/{validation_file}")
    validation_path = hf_hub_download(
        repo_id=data_repo,
        filename=validation_file,
        repo_type="dataset",
        token=token,
        local_dir=str(cache_dir / "dataset"),
    )
    return Path(ckpt_path), (Path(training_state_path) if training_state_path else None), Path(validation_path)


def load_breakpoints(training_state_path: Optional[Path]) -> Optional[List[float]]:
    if training_state_path is None or not training_state_path.exists():
        return None
    try:
        with training_state_path.open("r") as f:
            state = json.load(f)
        easiness_dict = state.get("easiness_dict")
        if isinstance(easiness_dict, dict):
            bp = easiness_dict.get("breakpoints")
            if bp:
                print(f"Loaded {len(bp)} easiness CDF breakpoints from training_state.json")
                return bp
    except Exception as exc:
        print(f"WARNING: failed to parse easiness breakpoints: {exc}")
    return None


def load_model(ckpt_path: Path, breakpoints: Optional[List[float]], dev: DeviceContext):
    config = HELMConfig(easiness_cdf_breakpoints=breakpoints)
    model = HELMForMaskedLM(config)
    print(f"Loading checkpoint from {ckpt_path}")
    payload = torch.load(str(ckpt_path), map_location="cpu")
    if not isinstance(payload, dict):
        raise RuntimeError("Checkpoint is not a dict")
    state = payload.get("model_state", payload)
    state = strip_state_prefixes(state)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing or unexpected:
        print(f"State load: {len(missing)} missing keys, {len(unexpected)} unexpected keys")
        if missing:
            print("  Missing (first 10):", missing[:10])
        if unexpected:
            print("  Unexpected (first 10):", unexpected[:10])
        # The analysis should not silently continue through a large mismatch.
        if len(missing) > 5 or len(unexpected) > 5:
            raise RuntimeError("Large state-dict mismatch; make sure this is HELM_7c model.py")
    del payload
    model.to(dev.device)
    model.eval()
    # Force the same fixed-shape dense attention backend used during training, then mask.
    model.enable_efficient_inference("dense", compile=False)
    return model, config


def deterministic_span_mask(ids: torch.Tensor, config: HELMConfig, seed: int,
                            probability: float = 0.30, span_length: int = 3) -> Tuple[torch.Tensor, torch.Tensor]:
    """Create one deterministic MLM example. Relative CE comparisons are the goal."""
    ids = ids.clone().long()
    labels = torch.full_like(ids, -100)
    g = torch.Generator(device="cpu")
    g.manual_seed(int(seed))

    special = {
        int(config.bos_token_id), int(config.eos_token_id), int(config.pad_token_id),
        int(config.mask_token_id), int(config.unk_token_id),
    }
    candidate = [i for i, tok in enumerate(ids.tolist()) if int(tok) not in special]
    if not candidate:
        return ids, labels

    target = max(1, int(round(probability * len(candidate))))
    candidate_set = set(candidate)
    perm = torch.randperm(len(candidate), generator=g).tolist()
    chosen = set()
    for pi in perm:
        if len(chosen) >= target:
            break
        start = candidate[pi]
        for pos in range(start, min(start + span_length, ids.numel())):
            if pos in candidate_set:
                chosen.add(pos)
                if len(chosen) >= target:
                    break

    chosen = sorted(chosen)
    if not chosen:
        chosen = [candidate[0]]
    pos = torch.tensor(chosen, dtype=torch.long)
    original = ids[pos].clone()
    labels[pos] = original

    r = torch.rand(len(pos), generator=g)
    mask_sel = r < 0.80
    random_sel = (r >= 0.80) & (r < 0.90)
    ids[pos[mask_sel]] = int(config.mask_token_id)
    if random_sel.any():
        random_tokens = torch.randint(
            low=0, high=int(config.vocab_size), size=(int(random_sel.sum()),), generator=g
        )
        ids[pos[random_sel]] = random_tokens
    # remaining 10% unchanged
    return ids, labels


def prepare_batches(validation_path: Path, config: HELMConfig, num_examples: int,
                    batch_size: int, seq_len: int, seed: int) -> List[Dict[str, torch.Tensor]]:
    table = pq.read_table(str(validation_path), columns=["input_ids", "easiness_score"])
    if "input_ids" not in table.column_names or "easiness_score" not in table.column_names:
        raise RuntimeError(f"Validation parquet columns are {table.column_names}; expected input_ids and easiness_score")

    total_rows = table.num_rows
    n = min(num_examples, total_rows)
    rng = np.random.default_rng(seed)
    indices = rng.permutation(total_rows)[:n]
    input_col = table.column("input_ids")
    easy_col = table.column("easiness_score")

    examples = []
    for i, row_idx in enumerate(indices.tolist()):
        ids_list = input_col[row_idx].as_py()
        easy_val = easy_col[row_idx].as_py()
        ids = torch.tensor(ids_list, dtype=torch.long)[:seq_len]
        if ids.numel() < seq_len:
            pad = torch.full((seq_len - ids.numel(),), int(config.pad_token_id), dtype=torch.long)
            ids = torch.cat([ids, pad], dim=0)
        masked, labels = deterministic_span_mask(ids, config, seed=seed + 100003 * i)
        examples.append({
            "input_ids": masked,
            "labels": labels,
            "attention_mask": (masked != int(config.pad_token_id)).long(),
            "easiness_score": torch.tensor(float(easy_val), dtype=torch.float32),
            "example_id": torch.tensor(i, dtype=torch.long),
        })

    # Drop an incomplete final batch to keep XLA/static shapes stable.
    usable = (len(examples) // batch_size) * batch_size
    examples = examples[:usable]
    if not examples:
        raise RuntimeError("Not enough examples for one complete batch")

    batches = []
    for start in range(0, len(examples), batch_size):
        chunk = examples[start:start + batch_size]
        batches.append({k: torch.stack([x[k] for x in chunk], dim=0) for k in chunk[0]})
    print(f"Prepared {len(examples)} examples -> {len(batches)} batches of {batch_size}, seq_len={seq_len}")
    return batches


def move_batch(batch: Dict[str, torch.Tensor], dev: DeviceContext) -> Dict[str, torch.Tensor]:
    return {k: v.to(dev.device) for k, v in batch.items()}


# -----------------------------------------------------------------------------
# CE helpers
# -----------------------------------------------------------------------------
def ce_sum_and_count(logits: torch.Tensor, labels: torch.Tensor, chunk_tokens: int = 128):
    """Fixed-shape chunked CE. Returns differentiable sum and integer-ish tensor count."""
    total = logits.new_zeros((), dtype=torch.float32)
    count = labels.new_zeros((), dtype=torch.long)
    seq_len = logits.size(1)
    vocab = logits.size(-1)
    for start in range(0, seq_len, chunk_tokens):
        end = min(start + chunk_tokens, seq_len)
        lgt = logits[:, start:end, :].float().reshape(-1, vocab)
        lab = labels[:, start:end].reshape(-1)
        total = total + F.cross_entropy(lgt, lab, ignore_index=-100, reduction="sum")
        count = count + (lab != -100).sum()
    return total, count


def forward_ce(model, batch: Dict[str, torch.Tensor], dev: DeviceContext,
               pass_easiness: bool = True, backward: bool = False):
    kwargs = {
        "input_ids": batch["input_ids"],
        "attention_mask": batch["attention_mask"],
        "current_step": 6500,
        "easiness_score": batch["easiness_score"] if pass_easiness else None,
    }
    with dev.autocast():
        logits, count_loss = model(**kwargs)
        ce_sum, ce_count = ce_sum_and_count(logits, batch["labels"])
        ce = ce_sum / ce_count.clamp_min(1).to(ce_sum.dtype)
    if backward:
        ce.backward()
        dev.mark_step()
    return ce, ce_sum, ce_count, count_loss


# -----------------------------------------------------------------------------
# Router mask override hooks
# -----------------------------------------------------------------------------
class RouterOverride:
    """Temporarily replace every router's [B,H,1,1] output without changing shapes."""
    def __init__(self, model, mode: str, permanent_heads: int, forced_masks: Optional[Dict[int, torch.Tensor]] = None):
        self.model = model
        self.mode = mode
        self.permanent_heads = permanent_heads
        self.forced_masks = forced_masks or {}
        self.handles = []
        self.gates: Dict[int, torch.Tensor] = {}

    def _hook(self, layer_idx: int):
        def hook(module, inputs, output):
            b, h, _, _ = output.shape
            p = self.permanent_heads
            if self.mode == "dense":
                return torch.ones_like(output)
            if self.mode == "permanent_only":
                result = torch.zeros_like(output)
                result[:, :p] = 1.0
                return result
            if self.mode == "random_same_count":
                elastic = output[:, p:, 0, 0]
                k = (elastic > 0.5).sum(dim=-1, keepdim=True)
                noise = torch.rand_like(elastic.float())
                ranks = double_argsort_rank(noise, descending=True)
                rand_elastic = (ranks < k).to(output.dtype)
                permanent = torch.ones((b, p), device=output.device, dtype=output.dtype)
                return torch.cat([permanent, rand_elastic], dim=-1).view(b, h, 1, 1)
            if self.mode == "forced":
                if layer_idx not in self.forced_masks:
                    return output
                mask = self.forced_masks[layer_idx].to(device=output.device, dtype=output.dtype)
                return mask.view(b, h, 1, 1)
            if self.mode == "gate_dense":
                gate = torch.ones_like(output, requires_grad=True)
                self.gates[layer_idx] = gate
                return gate
            if self.mode == "gate_routed":
                gate = torch.ones_like(output, requires_grad=True)
                self.gates[layer_idx] = gate
                return output * gate
            raise ValueError(f"Unknown override mode {self.mode}")
        return hook

    def __enter__(self):
        for i, block in enumerate(self.model.model.blocks):
            self.handles.append(block.mlt_vw_rtr.register_forward_hook(self._hook(i)))
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


# -----------------------------------------------------------------------------
# Router statistics
# -----------------------------------------------------------------------------
def collect_router_statistics(model, batches, dev: DeviceContext, output_dir: Path, selected_layers: Sequence[int]):
    n_layers = len(model.model.blocks)
    logits_by_layer = [[] for _ in range(n_layers)]
    sig_by_layer = [[] for _ in range(n_layers)]
    mask_by_layer = [[] for _ in range(n_layers)]
    actual_by_layer = [[] for _ in range(n_layers)]
    target_by_layer = [[] for _ in range(n_layers)]
    easiness_all = []
    ce_total = 0.0
    ce_tokens = 0

    with torch.no_grad():
        for bi, cpu_batch in enumerate(batches):
            batch = move_batch(cpu_batch, dev)
            ce, ce_sum, ce_count, _ = forward_ce(model, batch, dev, pass_easiness=True)
            dev.mark_step()
            ce_total += float(ce_sum.detach().cpu())
            ce_tokens += int(ce_count.detach().cpu())
            easiness_all.append(cpu_batch["easiness_score"].numpy())

            for li, block in enumerate(model.model.blocks):
                r = block.mlt_vw_rtr
                logits_by_layer[li].append(r.save_router_logits.detach().float().cpu().numpy())
                sig_by_layer[li].append(r.save_sigmoid_scores.detach().float().cpu().numpy())
                mask_by_layer[li].append(r.save_hard_mask.detach().float().cpu().numpy())
                actual_by_layer[li].append(r.save_total_head_count.detach().float().cpu().numpy())
                target_by_layer[li].append(r.save_target_total_head_count.detach().float().cpu().numpy())

    easiness = np.concatenate(easiness_all, axis=0)
    per_layer = {}
    calibration_rows = []

    for li in range(n_layers):
        logits = np.concatenate(logits_by_layer[li], axis=0)
        sig = np.concatenate(sig_by_layer[li], axis=0)
        mask = np.concatenate(mask_by_layer[li], axis=0)
        actual = np.concatenate(actual_by_layer[li], axis=0)
        target = np.concatenate(target_by_layer[li], axis=0)
        freq = mask.mean(axis=0)
        weight_norms = (
            model.model.blocks[li].mlt_vw_rtr.q_up_proj.weight.detach().float().norm(dim=1).cpu().numpy()
        )

        stats = {
            "actual_mean": float(actual.mean()),
            "target_mean": float(target.mean()),
            "count_mae": float(np.abs(actual - target).mean()),
            "count_error_mean": float((actual - target).mean()),
            "easiness_actual_spearman": spearman_np(easiness, actual),
            "target_actual_spearman": spearman_np(target, actual),
            "dynamic_head_fraction_05_95": float(((freq > 0.05) & (freq < 0.95)).mean()),
            "always_off_fraction_lt_01": float((freq < 0.01).mean()),
            "always_on_fraction_gt_99": float((freq > 0.99).mean()),
            "expected_pairwise_mask_hamming": float(np.mean(2.0 * freq * (1.0 - freq))),
            "logit_input_variance_mean": float(logits.var(axis=0).mean()),
            "logit_across_head_variance_mean": float(logits.var(axis=1).mean()),
            "sigmoid_mean": float(sig.mean()),
            "sigmoid_saturation_lt05_gt95": float(((sig < 0.05) | (sig > 0.95)).mean()),
            "sigmoid_near_half_45_55": float(((sig > 0.45) & (sig < 0.55)).mean()),
            "ste_derivative_mean": float((sig * (1.0 - sig)).mean()),
            "router_weight_norm_mean": float(weight_norms.mean()),
            "router_weight_norm_std": float(weight_norms.std()),
            "min_total_head_fraction": float((actual == model.config.head_target_min).mean()),
            "max_total_head_fraction": float((actual == model.config.head_target_max).mean()),
        }
        per_layer[li] = stats

        rows = []
        for h in range(mask.shape[1]):
            rows.append({
                "layer": li,
                "elastic_head": h,
                "absolute_head": h + model.config.num_permanent_heads,
                "activation_frequency": float(freq[h]),
                "mean_logit": float(logits[:, h].mean()),
                "std_logit_across_examples": float(logits[:, h].std()),
                "mean_sigmoid": float(sig[:, h].mean()),
                "router_weight_norm": float(weight_norms[h]),
            })
        with (output_dir / f"layer_{li:02d}_head_stats.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader(); writer.writerows(rows)

        # Count calibration by integer target.
        for t in range(int(model.config.head_target_min), int(model.config.head_target_max) + 1):
            idx = target == t
            if idx.any():
                calibration_rows.append({
                    "layer": li,
                    "target_heads": t,
                    "n": int(idx.sum()),
                    "actual_mean": float(actual[idx].mean()),
                    "actual_std": float(actual[idx].std()),
                    "mae": float(np.abs(actual[idx] - target[idx]).mean()),
                    "fraction_exact": float((actual[idx] == target[idx]).mean()),
                })

        if li in selected_layers:
            save_heatmap(
                output_dir / f"layer_{li:02d}_activation_heatmap.png",
                mask, f"Layer {li}: elastic hard activations", "Elastic head", "Example", 0.0, 1.0,
            )
            save_heatmap(
                output_dir / f"layer_{li:02d}_sigmoid_heatmap.png",
                sig, f"Layer {li}: sigmoid scores", "Elastic head", "Example", 0.0, 1.0,
            )
            save_bar(
                output_dir / f"layer_{li:02d}_activation_frequency.png",
                freq, f"Layer {li}: elastic activation frequency", "Activation frequency",
            )
            save_bar(
                output_dir / f"layer_{li:02d}_router_weight_norms.png",
                weight_norms, f"Layer {li}: router row norms", "L2 norm",
            )

    if calibration_rows:
        with (output_dir / "count_calibration.csv").open("w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=calibration_rows[0].keys())
            writer.writeheader(); writer.writerows(calibration_rows)

    return {
        "routed_ce": ce_total / max(1, ce_tokens),
        "easiness": easiness,
        "per_layer": per_layer,
        "logits": [np.concatenate(x, axis=0) for x in logits_by_layer],
        "sigmoid": [np.concatenate(x, axis=0) for x in sig_by_layer],
        "masks": [np.concatenate(x, axis=0) for x in mask_by_layer],
        "actual": [np.concatenate(x, axis=0) for x in actual_by_layer],
        "target": [np.concatenate(x, axis=0) for x in target_by_layer],
    }


# -----------------------------------------------------------------------------
# CE mode comparisons
# -----------------------------------------------------------------------------
def evaluate_override_mode(model, batches, dev: DeviceContext, mode: str, random_seed: int = 0) -> float:
    total_sum = 0.0
    total_count = 0
    seed_everything(random_seed)
    manager = contextlib.nullcontext() if mode == "routed" else RouterOverride(
        model, mode, model.config.num_permanent_heads
    )
    with manager:
        with torch.no_grad():
            for cpu_batch in batches:
                batch = move_batch(cpu_batch, dev)
                _, ce_sum, ce_count, _ = forward_ce(model, batch, dev, pass_easiness=False)
                dev.mark_step()
                total_sum += float(ce_sum.detach().cpu())
                total_count += int(ce_count.detach().cpu())
    return total_sum / max(1, total_count)


# -----------------------------------------------------------------------------
# Utility / alignment analysis
# -----------------------------------------------------------------------------
def build_topk_mask_from_importance(importance: torch.Tensor, baseline_full_mask: torch.Tensor,
                                    permanent_heads: int) -> torch.Tensor:
    """Keep permanent heads + same elastic count as HELM, but choose highest-importance elastic heads."""
    p = permanent_heads
    imp_e = importance[:, p:]
    base_e = baseline_full_mask[:, p:]
    k = (base_e > 0.5).sum(dim=-1, keepdim=True)
    ranks = double_argsort_rank(imp_e, descending=True)
    chosen = (ranks < k).to(importance.dtype)
    permanent = torch.ones((importance.size(0), p), device=importance.device, dtype=importance.dtype)
    return torch.cat([permanent, chosen], dim=-1)

def utility_alignment_analysis(model, batches, dev: DeviceContext, max_batches: int, output_dir: Path):
    """
    Measure whether HELM's router logits agree with the heads that CE actually benefits from.

    For a temporary dense gate g_h = 1:

        dL/dg_h < 0  -> increasing/keeping the head lowers loss
        dL/dg_h > 0  -> removing the head is predicted to lower loss

    Therefore define signed first-order utility:

        utility_h = -dL/dg_h

    Positive utility = beneficial to keep.
    Negative utility = potentially harmful.
    """

    # Freeze actual model parameters. Backward is ONLY for temporary gates.
    original_requires_grad = {id(p): p.requires_grad for p in model.parameters()}
    for p in model.parameters():
        p.requires_grad_(False)

    n_layers = len(model.model.blocks)
    rows = []

    oracle_ce_sum = 0.0
    oracle_tokens = 0

    subset_routed_sum = 0.0
    subset_routed_tokens = 0

    dense_gate_sum = 0.0
    dense_gate_tokens = 0

    for bi, cpu_batch in enumerate(batches[:max_batches]):
        batch = move_batch(cpu_batch, dev)

        # ----------------------------------------------------------
        # 1. Normal HELM routed pass.
        #    Save router logits and actual selected elastic heads.
        # ----------------------------------------------------------
        with torch.no_grad():
            _, rsum, rcount, _ = forward_ce(
                model,
                batch,
                dev,
                pass_easiness=True,
            )
            dev.mark_step()

            subset_routed_sum += float(rsum.detach().cpu())
            subset_routed_tokens += int(rcount.detach().cpu())

            baseline_logits = [
                block.mlt_vw_rtr.save_router_logits.detach().float().cpu().numpy()
                for block in model.model.blocks
            ]

            baseline_elastic = [
                block.mlt_vw_rtr.save_hard_mask.detach().float().cpu().numpy()
                for block in model.model.blocks
            ]

        # ----------------------------------------------------------
        # 2. Force all heads ON, but replace each layer's mask with
        #    differentiable temporary gates.
        #
        #    One backward gives dCE/dgate for all heads/layers.
        # ----------------------------------------------------------
        model.zero_grad(set_to_none=True)

        with RouterOverride(
            model,
            "gate_dense",
            model.config.num_permanent_heads,
        ) as gate_override:

            ce, dsum, dcount, _ = forward_ce(
                model,
                batch,
                dev,
                pass_easiness=False,
                backward=True,
            )

            dense_gate_sum += float(dsum.detach().cpu())
            dense_gate_tokens += int(dcount.detach().cpu())

            utility_by_layer: Dict[int, torch.Tensor] = {}
            impact_by_layer: Dict[int, torch.Tensor] = {}

            for li in range(n_layers):
                gate = gate_override.gates[li]

                if gate.grad is None:
                    raise RuntimeError(
                        f"No gradient for temporary gate at layer {li}"
                    )

                grad = gate.grad.detach()[:, :, 0, 0].float()

                # SIGNED first-order benefit of KEEPING the head.
                #
                # Removing: Δg = -1
                #
                # ΔL ≈ grad * (-1) = -grad
                #
                # Positive utility => removal increases CE.
                utility_by_layer[li] = -grad

                # Absolute gradient is still useful, but only as
                # sensitivity / impact -- NOT as directional utility.
                impact_by_layer[li] = grad.abs()

        # ----------------------------------------------------------
        # 3. Construct same-cardinality first-order oracle:
        #
        #    keep HELM's exact elastic count K,
        #    but choose the K heads with highest SIGNED utility.
        # ----------------------------------------------------------
        forced_masks = {}

        for li in range(n_layers):
            p = model.config.num_permanent_heads

            elastic_np = baseline_elastic[li]

            full_base = torch.cat(
                [
                    torch.ones(
                        (elastic_np.shape[0], p),
                        dtype=torch.float32,
                    ),
                    torch.tensor(
                        elastic_np,
                        dtype=torch.float32,
                    ),
                ],
                dim=-1,
            ).to(dev.device)

            forced_masks[li] = build_topk_mask_from_importance(
                utility_by_layer[li],
                full_base,
                p,
            ).detach()

            # ------------------------------------------------------
            # Per-example router-vs-utility statistics.
            # ------------------------------------------------------
            logits_np = baseline_logits[li]

            signed_util_np = (
                utility_by_layer[li][:, p:]
                .detach()
                .cpu()
                .numpy()
            )

            impact_np = (
                impact_by_layer[li][:, p:]
                .detach()
                .cpu()
                .numpy()
            )

            base_np = elastic_np

            for ex in range(logits_np.shape[0]):
                k = int(base_np[ex].sum())

                # Does a larger HELM router logit mean larger
                # actual CE benefit?
                corr = spearman_np(
                    logits_np[ex],
                    signed_util_np[ex],
                )

                selected = np.flatnonzero(
                    base_np[ex] > 0.5
                )

                inactive = np.flatnonzero(
                    base_np[ex] <= 0.5
                )

                # ----------------------------------------------
                # Top-K overlap using SIGNED utility.
                # ----------------------------------------------
                if k > 0:
                    top_idx = np.argsort(
                        -signed_util_np[ex]
                    )[:k]

                    overlap = (
                        len(
                            set(top_idx.tolist())
                            & set(selected.tolist())
                        )
                        / float(k)
                    )
                else:
                    overlap = 1.0

                # ----------------------------------------------
                # Signed CE utility.
                # ----------------------------------------------
                selected_utility = (
                    float(
                        signed_util_np[ex][selected].mean()
                    )
                    if len(selected)
                    else float("nan")
                )

                inactive_utility = (
                    float(
                        signed_util_np[ex][inactive].mean()
                    )
                    if len(inactive)
                    else float("nan")
                )

                utility_gap = (
                    selected_utility - inactive_utility
                    if (
                        np.isfinite(selected_utility)
                        and np.isfinite(inactive_utility)
                    )
                    else float("nan")
                )

                # ----------------------------------------------
                # Absolute sensitivity, retained separately.
                # ----------------------------------------------
                selected_impact = (
                    float(
                        impact_np[ex][selected].mean()
                    )
                    if len(selected)
                    else float("nan")
                )

                inactive_impact = (
                    float(
                        impact_np[ex][inactive].mean()
                    )
                    if len(inactive)
                    else float("nan")
                )

                impact_ratio = (
                    selected_impact
                    / (inactive_impact + 1e-12)
                    if (
                        np.isfinite(selected_impact)
                        and np.isfinite(inactive_impact)
                    )
                    else float("nan")
                )

                # Fraction of heads where first-order analysis
                # says keeping the head is actually beneficial.
                positive_utility_fraction = float(
                    (signed_util_np[ex] > 0).mean()
                )

                rows.append(
                    {
                        "batch": bi,
                        "example_in_batch": ex,
                        "layer": li,
                        "elastic_k": k,

                        "logit_signed_utility_spearman": corr,
                        "router_topk_overlap_with_signed_utility": overlap,

                        "selected_signed_utility_mean": selected_utility,
                        "inactive_signed_utility_mean": inactive_utility,
                        "selected_minus_inactive_signed_utility": utility_gap,

                        "selected_abs_gradient_mean": selected_impact,
                        "inactive_abs_gradient_mean": inactive_impact,
                        "selected_to_inactive_abs_gradient_ratio": impact_ratio,

                        "positive_signed_utility_fraction": positive_utility_fraction,
                    }
                )

        # ----------------------------------------------------------
        # 4. Evaluate the same-count signed-utility oracle.
        # ----------------------------------------------------------
        with RouterOverride(
            model,
            "forced",
            model.config.num_permanent_heads,
            forced_masks=forced_masks,
        ):
            with torch.no_grad():
                _, osum, ocount, _ = forward_ce(
                    model,
                    batch,
                    dev,
                    pass_easiness=False,
                )
                dev.mark_step()

                oracle_ce_sum += float(osum.detach().cpu())
                oracle_tokens += int(ocount.detach().cpu())

    # Restore parameter flags.
    for p in model.parameters():
        p.requires_grad_(
            original_requires_grad[id(p)]
        )

    # --------------------------------------------------------------
    # Save raw per-example results.
    # --------------------------------------------------------------
    if rows:
        with (
            output_dir / "utility_alignment.csv"
        ).open("w", newline="") as f:

            writer = csv.DictWriter(
                f,
                fieldnames=rows[0].keys(),
            )
            writer.writeheader()
            writer.writerows(rows)

    # --------------------------------------------------------------
    # Aggregate per layer.
    # --------------------------------------------------------------
    per_layer = {}

    for li in range(n_layers):
        rr = [
            r for r in rows
            if r["layer"] == li
        ]

        def finite_mean(key):
            vals = np.array(
                [r[key] for r in rr],
                dtype=np.float64,
            )
            vals = vals[np.isfinite(vals)]

            return (
                float(vals.mean())
                if vals.size
                else float("nan")
            )

        per_layer[li] = {
            "logit_signed_utility_spearman_mean":
                finite_mean(
                    "logit_signed_utility_spearman"
                ),

            "router_topk_overlap_with_signed_utility_mean":
                finite_mean(
                    "router_topk_overlap_with_signed_utility"
                ),

            "selected_minus_inactive_signed_utility_mean":
                finite_mean(
                    "selected_minus_inactive_signed_utility"
                ),

            "selected_to_inactive_abs_gradient_ratio_mean":
                finite_mean(
                    "selected_to_inactive_abs_gradient_ratio"
                ),

            "positive_signed_utility_fraction_mean":
                finite_mean(
                    "positive_signed_utility_fraction"
                ),
        }

    return {
        "subset_routed_ce":
            subset_routed_sum / max(1, subset_routed_tokens),
    
        "dense_gate_ce":
            dense_gate_sum / max(1, dense_gate_tokens),
    
        "first_order_signed_oracle_same_count_ce":
            oracle_ce_sum / max(1, oracle_tokens),
    
        "per_layer": per_layer,
    }


# -----------------------------------------------------------------------------
# Functional redundancy analysis
# -----------------------------------------------------------------------------
class AttentionInputCapture:
    def __init__(self, model, layers: Sequence[int]):
        self.model = model
        self.layers = set(layers)
        self.handles = []
        self.data: Dict[int, Tuple[torch.Tensor, torch.Tensor]] = {}

    def _hook(self, li: int):
        def hook(module, inputs):
            hidden_states, attention_mask, router_mask = inputs
            self.data[li] = (hidden_states.detach(), attention_mask.detach())
        return hook

    def __enter__(self):
        for li in self.layers:
            self.handles.append(
                self.model.model.blocks[li].attn.register_forward_pre_hook(self._hook(li))
            )
        return self

    def __exit__(self, exc_type, exc, tb):
        for h in self.handles:
            h.remove()
        self.handles.clear()


def unmasked_attention_context(attn, hidden_states: torch.Tensor, attention_mask: torch.Tensor):
    """Reproduce HELM_7c dense attention up through pre-router head contexts."""
    qkv_proj = cast_linear(hidden_states, attn.qkv)
    b, s, _ = hidden_states.shape
    q, k, v = qkv_proj.split(attn.total_head_dim, dim=-1)
    q = q.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    k = k.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    v = v.view(b, s, attn.num_attention_heads, attn.d_head).permute(0, 2, 1, 3)
    q = justnorm(q)
    k = justnorm(k)
    q = attn.RoPE(q)
    k = attn.RoPE(k)
    sqk = attn.sqk * (attn.ngpt_sqk_init_value / attn.ngpt_sqk_init_scale)
    sqk = sqk.view(1, attn.num_attention_heads, 1, attn.d_head).to(q.dtype)
    q = sqk * q
    k = sqk * k
    context = F.scaled_dot_product_attention(
        q, k, v,
        attn_mask=attention_mask.to(q.dtype),
        scale=math.sqrt(attn.d_head),
    )
    if attn.config.use_exclusive_attention:
        vn = F.normalize(v, dim=-1)
        context = context - (context * vn).sum(dim=-1, keepdim=True) * vn
    return context


def parameter_head_cosine(attn) -> np.ndarray:
    h, d = attn.num_attention_heads, attn.d_head
    D = attn.hidden_size
    qkv = attn.qkv.weight.detach().float().cpu()
    q, k, v = qkv.split(attn.total_head_dim, dim=0)
    q = q.view(h, d, D).reshape(h, -1)
    k = k.view(h, d, D).reshape(h, -1)
    v = v.view(h, d, D).reshape(h, -1)
    # output.weight = [D, H*d]; each head owns one contiguous input-column block.
    o = attn.output.weight.detach().float().cpu().view(D, h, d).permute(1, 0, 2).reshape(h, -1)
    vec = torch.cat([q, k, v, o], dim=1)
    vec = F.normalize(vec, dim=1)
    return (vec @ vec.T).numpy()


def functional_redundancy_analysis(model, batches, dev: DeviceContext, layers: Sequence[int],
                                   max_batches: int, sample_tokens: int, output_dir: Path):
    grams = {li: np.zeros((model.config.num_attention_heads, model.config.num_attention_heads), dtype=np.float64)
             for li in layers}
    sample_count = {li: 0 for li in layers}

    with torch.no_grad():
        for bi, cpu_batch in enumerate(batches[:max_batches]):
            batch = move_batch(cpu_batch, dev)
            with AttentionInputCapture(model, layers) as cap:
                with dev.autocast():
                    _ = model(
                        input_ids=batch["input_ids"],
                        attention_mask=batch["attention_mask"],
                        current_step=6500,
                        easiness_score=batch["easiness_score"],
                    )
                dev.mark_step()

            for li in layers:
                hidden, attn_mask = cap.data[li]
                attn = model.model.blocks[li].attn
                with dev.autocast():
                    context = unmasked_attention_context(attn, hidden, attn_mask)  # [B,H,S,d]
                    s = context.size(2)
                    t = min(sample_tokens, s)
                    # Deterministic, spread-out positions to avoid only sampling one local region.
                    positions = torch.linspace(0, s - 1, steps=t, device=context.device).long()
                    c = context.index_select(2, positions)  # [B,H,T,d]
                    W = attn.output.weight.to(c.dtype).view(attn.hidden_size, attn.num_attention_heads, attn.d_head)
                    # Per-head residual contribution Y_h = C_h @ W_O^(h)^T.
                    y = torch.einsum("bhtd,ohd->bhto", c, W)  # [B,H,T,D]
                    flat = y.permute(1, 0, 2, 3).contiguous().view(attn.num_attention_heads, -1).float()
                    gram = flat @ flat.T
                dev.mark_step()
                grams[li] += gram.detach().cpu().float().numpy().astype(np.float64)
                sample_count[li] += int(flat.shape[1])
                del context, c, y, flat, gram

    results = {}

    P = model.config.num_permanent_heads
    H = model.config.num_attention_heads

    for li in layers:
        gram = grams[li]

        # ----------------------------------------------------------
        # Whole-layer similarity/rank
        # ----------------------------------------------------------
        functional_cos = cosine_from_gram(gram)

        all_erank, all_prank, eig = effective_rank_from_gram(
            gram
        )

        # ----------------------------------------------------------
        # Permanent-head subspace
        # ----------------------------------------------------------
        permanent_gram = gram[:P, :P]

        perm_erank, perm_prank, perm_eig = (
            effective_rank_from_gram(
                permanent_gram
            )
        )

        # ----------------------------------------------------------
        # Elastic-head subspace
        # ----------------------------------------------------------
        elastic_gram = gram[P:, P:]

        elastic_erank, elastic_prank, elastic_eig = (
            effective_rank_from_gram(
                elastic_gram
            )
        )

        # ----------------------------------------------------------
        # Per-head residual-stream RMS.
        #
        # gram[h,h] = sum(Y_h ** 2)
        # sample_count = number of scalar samples accumulated
        # ----------------------------------------------------------
        denom = max(1, sample_count[li])

        head_energy = np.clip(
            np.diag(gram),
            0.0,
            None,
        )

        head_rms = np.sqrt(
            head_energy / float(denom)
        )

        total_energy = (
            head_energy.sum() + 1e-12
        )

        head_energy_fraction = (
            head_energy / total_energy
        )

        # ----------------------------------------------------------
        # Parameter-space comparison
        # ----------------------------------------------------------
        param_cos = parameter_head_cosine(
            model.model.blocks[li].attn
        )

        fstats = offdiag_stats(
            functional_cos
        )

        pstats = offdiag_stats(
            param_cos
        )

        results[li] = {
            # All heads
            "functional_entropy_effective_rank":
                all_erank,

            "functional_participation_rank":
                all_prank,

            # Permanent 8 only
            "permanent_entropy_effective_rank":
                perm_erank,

            "permanent_participation_rank":
                perm_prank,

            # Elastic 24 only
            "elastic_entropy_effective_rank":
                elastic_erank,

            "elastic_participation_rank":
                elastic_prank,

            # Per-head energy
            "head_residual_rms":
                head_rms,

            "head_energy_fraction":
                head_energy_fraction,

            "functional_similarity":
                fstats,

            "parameter_similarity":
                pstats,

            "top_functional_pairs":
                top_pairs(
                    functional_cos,
                    8,
                ),

            "top_parameter_pairs":
                top_pairs(
                    param_cos,
                    8,
                ),
        }

        # ----------------------------------------------------------
        # Save per-head residual energy table.
        # ----------------------------------------------------------
        rms_rows = []

        for h in range(H):
            rms_rows.append(
                {
                    "head": h,

                    "group":
                        "permanent"
                        if h < P
                        else "elastic",

                    "residual_rms":
                        float(head_rms[h]),

                    "energy_fraction":
                        float(
                            head_energy_fraction[h]
                        ),
                }
            )

        with (
            output_dir
            / f"layer_{li:02d}_head_residual_energy.csv"
        ).open("w", newline="") as f:

            writer = csv.DictWriter(
                f,
                fieldnames=rms_rows[0].keys(),
            )

            writer.writeheader()
            writer.writerows(rms_rows)

        save_matrix_csv(
            output_dir
            / f"layer_{li:02d}_functional_similarity.csv",
            functional_cos,
        )

        save_matrix_csv(
            output_dir
            / f"layer_{li:02d}_parameter_similarity.csv",
            param_cos,
        )

        save_heatmap(
            output_dir / f"layer_{li:02d}_functional_similarity.png",
            functional_cos,
            f"Layer {li}: cosine of unmasked residual head contributions",
            "Head", "Head", -1.0, 1.0,
        )
        save_heatmap(
            output_dir / f"layer_{li:02d}_parameter_similarity.png",
            param_cos,
            f"Layer {li}: cosine of concatenated Q/K/V/O head parameters",
            "Head", "Head", -1.0, 1.0,
        )

        # Eigen spectrum plot.
        vals = np.sort(np.clip(eig, 0, None))[::-1]
        frac = vals / (vals.sum() + 1e-12)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.plot(np.arange(1, len(frac) + 1), frac, marker="o")
        ax.set_title(f"Layer {li}: head-contribution Gram eigenvalue spectrum")
        ax.set_xlabel("Component")
        ax.set_ylabel("Fraction of head-space energy")
        fig.tight_layout()
        fig.savefig(output_dir / f"layer_{li:02d}_functional_eigenspectrum.png", dpi=160)
        plt.close(fig)

    return results


# -----------------------------------------------------------------------------
# Optional exact single-head ablation
# -----------------------------------------------------------------------------
def exact_ablation_analysis(model, batches, dev: DeviceContext, layers: Sequence[int], max_batches: int,
                            output_dir: Path):
    """Slow but interpretable: from forced-dense mode, remove one head in one selected layer."""
    subset = batches[:max_batches]
    dense_ce = evaluate_override_mode(model, subset, dev, "dense")
    rows = []
    H = model.config.num_attention_heads

    for li in layers:
        for head in range(H):
            total_sum = 0.0
            total_count = 0
            for cpu_batch in subset:
                batch = move_batch(cpu_batch, dev)
                b = batch["input_ids"].size(0)
                # All layers dense; one layer gets dense-minus-one-head.
                forced = {}
                for lj in range(len(model.model.blocks)):
                    m = torch.ones((b, H), dtype=torch.float32, device=dev.device)
                    if lj == li:
                        m[:, head] = 0.0
                    forced[lj] = m
                with RouterOverride(model, "forced", model.config.num_permanent_heads, forced_masks=forced):
                    with torch.no_grad():
                        _, s, c, _ = forward_ce(model, batch, dev, pass_easiness=False)
                        dev.mark_step()
                        total_sum += float(s.detach().cpu())
                        total_count += int(c.detach().cpu())
            ce = total_sum / max(1, total_count)
            rows.append({
                "layer": li,
                "head": head,
                "dense_ce": dense_ce,
                "ablated_ce": ce,
                "delta_ce": ce - dense_ce,
            })
            print(f"Exact ablation layer {li} head {head:02d}: ΔCE={ce-dense_ce:+.6f}")

    with (output_dir / "exact_single_head_ablation.csv").open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader(); writer.writerows(rows)
    return {"dense_ce_subset": dense_ce, "rows": rows}


# -----------------------------------------------------------------------------
# Reporting
# -----------------------------------------------------------------------------
def json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, float) and not np.isfinite(obj):
        return None
    return obj


def write_summary(output_dir: Path, args, router_stats, ce_modes, utility, redundancy, exact):
    lines = []
    lines.append("# HELM_7c Head Dynamics Analysis\n")
    lines.append(f"Checkpoint: `{args.model_repo}/{args.checkpoint}`\n")
    lines.append(f"Examples: {args.num_examples}, sequence length: {args.seq_len}, batch size: {args.batch_size}\n")
    lines.append("\n## 1. CE mode comparisons\n")
    for k, v in ce_modes.items():
        lines.append(f"- **{k}**: {v:.6f}\n")
    if utility:
        lines.append(f"- **utility subset routed CE**: {utility['subset_routed_ce']:.6f}\n")
        lines.append(f"- **utility subset forced-dense CE**: {utility['dense_gate_ce']:.6f}\n")
        lines.append(
            f"- **signed first-order oracle same-count CE**: "
            f"{utility['first_order_signed_oracle_same_count_ce']:.6f}\n"
        )
    lines.append("\nInterpretation:\n")
    routed = ce_modes.get("routed")
    dense = ce_modes.get("forced_dense")
    random_ce = ce_modes.get("random_same_count_mean")
    perm = ce_modes.get("permanent_only")
    if routed is not None and dense is not None:
        if dense + 1e-4 < routed:
            lines.append("- Forced-dense is better than routed: the current sparse decisions are removing useful attention contribution.\n")
        elif routed + 1e-4 < dense:
            lines.append("- Routed is better than forced-dense: simply restoring every head does not improve this checkpoint.\n")
        else:
            lines.append("- Routed and forced-dense CE are nearly identical on this diagnostic set.\n")
    if routed is not None and random_ce is not None:
        if routed + 1e-4 < random_ce:
            lines.append("- Routed beats random-same-count: head identity selection contains useful information beyond cardinality.\n")
        else:
            lines.append("- Routed does not clearly beat random-same-count: router identity selection may be weak or heads may be redundant.\n")
    if perm is not None:
        lines.append(f"- Permanent-only gap vs routed: {perm-routed:+.6f} CE.\n")

    lines.append("\n## 2. Router/cardinality summary\n")
    for li in range(len(router_stats["per_layer"])):
        s = router_stats["per_layer"][li]
        lines.append(
            f"- L{li:02d}: actual={s['actual_mean']:.2f}, target={s['target_mean']:.2f}, "
            f"MAE={s['count_mae']:.2f}, rho(easy,count)={s['easiness_actual_spearman']:.3f}, "
            f"dynamic_heads={100*s['dynamic_head_fraction_05_95']:.1f}%, "
            f"mask_hamming={s['expected_pairwise_mask_hamming']:.3f}, "
            f"sat={100*s['sigmoid_saturation_lt05_gt95']:.1f}%, "
            f"STE'={s['ste_derivative_mean']:.4f}, Wnorm={s['router_weight_norm_mean']:.2f}\n"
        )

    lines.append("\n## 3. Router vs CE head utility\n")
    if utility:
        for li in range(len(utility["per_layer"])):
            u = utility["per_layer"][li]

            lines.append(
                f"- L{li:02d}: "
                f"rho(logit, signed CE utility)="
                f"{u['logit_signed_utility_spearman_mean']:.3f}, "
                f"top-k overlap="
                f"{u['router_topk_overlap_with_signed_utility_mean']:.3f}, "
                f"selected-inactive signed utility="
                f"{u['selected_minus_inactive_signed_utility_mean']:.6f}, "
                f"selected/inactive |gradient|="
                f"{u['selected_to_inactive_abs_gradient_ratio_mean']:.3f}, "
                f"positive utility fraction="
                f"{u['positive_signed_utility_fraction_mean']:.3f}\n"
            )

        oracle = utility[
            "first_order_signed_oracle_same_count_ce"
        ]

        subset = utility[
            "subset_routed_ce"
        ]

        if oracle + 1e-4 < subset:
            lines.append(
                "\nThe signed first-order same-count oracle beats "
                "HELM routing on this subset. This suggests there are "
                "head selections with the same compute budget that "
                "the current router is not identifying.\n"
            )

        elif subset + 1e-4 < oracle:
            lines.append(
                "\nHELM routing beats the signed first-order oracle. "
                "The local gradient approximation is not sufficient "
                "to improve the selected set, so exact intervention "
                "results should be trusted more heavily.\n"
            )

        else:
            lines.append(
                "\nHELM and the signed first-order same-count oracle "
                "are nearly identical on this subset.\n"
            )
    else:
        lines.append("Utility analysis was skipped.\n")
        
    lines.append("\n## 4. Functional redundancy\n")

    P = 8
    E = 24

    for li, r in redundancy.items():
        f = r["functional_similarity"]
        p = r["parameter_similarity"]

        rms = np.asarray(
            r["head_residual_rms"],
            dtype=np.float64,
        )

        energy = np.asarray(
            r["head_energy_fraction"],
            dtype=np.float64,
        )

        top_energy = np.argsort(
            -energy
        )[:5]

        top_energy_str = ", ".join(
            [
                f"h{h}={100*energy[h]:.1f}%"
                for h in top_energy
            ]
        )

        lines.append(
            f"- L{li:02d}:\n"
            f"  - all heads entropy-rank="
            f"{r['functional_entropy_effective_rank']:.2f}/32, "
            f"participation-rank="
            f"{r['functional_participation_rank']:.2f}/32\n"
            f"  - permanent heads entropy-rank="
            f"{r['permanent_entropy_effective_rank']:.2f}/{P}, "
            f"participation-rank="
            f"{r['permanent_participation_rank']:.2f}/{P}\n"
            f"  - elastic heads entropy-rank="
            f"{r['elastic_entropy_effective_rank']:.2f}/{E}, "
            f"participation-rank="
            f"{r['elastic_participation_rank']:.2f}/{E}\n"
            f"  - mean permanent RMS="
            f"{rms[:P].mean():.6f}\n"
            f"  - mean elastic RMS="
            f"{rms[P:].mean():.6f}\n"
            f"  - top residual-energy heads: "
            f"{top_energy_str}\n"
            f"  - mean |functional cosine|="
            f"{f['mean_abs']:.3f}, "
            f"frac |cos|>.8="
            f"{100*f['frac_abs_gt_0.8']:.1f}%, "
            f"mean |parameter cosine|="
            f"{p['mean_abs']:.3f}\n"
        )

    lines.append("\n### How to use this for the proposed nonlinear 1024→2048 refinement\n")
    lines.append(
        "- If functional effective rank is low / output similarities are high even though parameter similarities are modest, "
        "the expanded attention head bank is producing redundant residual features. That is evidence worth testing a nonlinear "
        "head-refinement/expansion ablation.\n"
    )
    lines.append(
        "- If head outputs are diverse but router-logit/utility alignment is poor and random-same-count is competitive, the "
        "attention expansion is probably not the first bottleneck; the selector is.\n"
    )
    lines.append(
        "- If head outputs are diverse and router selection is good, but dense still beats routed, the 0/1 mixture may be too "
        "coarse; that is the strongest case for testing weighted active-head contributions next.\n"
    )

    if exact is not None:
        lines.append("\n## 5. Exact dense single-head ablations\n")
        deltas = np.array([r["delta_ce"] for r in exact["rows"]])
        lines.append(f"Mean ΔCE: {deltas.mean():+.6f}; max ΔCE: {deltas.max():+.6f}; min ΔCE: {deltas.min():+.6f}\n")

    (output_dir / "summary.md").write_text("".join(lines))


def main():
    parser = argparse.ArgumentParser(description="Analyze HELM_7c checkpoint head dynamics")
    parser.add_argument("--model-repo", default=MODEL_REPO)
    parser.add_argument("--checkpoint", default=CHECKPOINT_FILE)
    parser.add_argument("--data-repo", default=DATA_REPO)
    parser.add_argument("--validation-file", default=VALIDATION_FILE)
    parser.add_argument("--device", choices=["auto", "cuda", "xla", "cpu"], default="auto")
    parser.add_argument("--num-examples", type=int, default=16,
                        help="Diagnostic examples. 16 is enough for first pass; use 32/64 for stronger statistics.")
    parser.add_argument("--batch-size", type=int, default=2)
    parser.add_argument("--seq-len", type=int, default=1024)
    parser.add_argument("--layers", default="0,5,11", help="Layers for functional redundancy plots")
    parser.add_argument("--utility-batches", type=int, default=4,
                        help="Batches for backward gate-importance + same-count oracle")
    parser.add_argument("--functional-batches", type=int, default=4)
    parser.add_argument("--functional-sample-tokens", type=int, default=16)
    parser.add_argument("--random-trials", type=int, default=3)
    parser.add_argument("--exact-ablation", action="store_true",
                        help="Slow: dense-minus-one-head exact CE for selected layers")
    parser.add_argument("--exact-ablation-batches", type=int, default=2)
    parser.add_argument("--seed", type=int, default=67)
    parser.add_argument("--cache-dir", default="./helm7c_analysis_cache")
    parser.add_argument("--output-dir", default="./helm7c_head_analysis")
    args = parser.parse_args()

    selected_layers = [int(x.strip()) for x in args.layers.split(",") if x.strip()]
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir = Path(args.cache_dir)
    seed_everything(args.seed)

    print("=" * 80)
    print("HELM_7c HEAD DYNAMICS ANALYSIS")
    print("=" * 80)
    print("This is a read-only checkpoint analysis. Model parameters will not be updated.\n")

    token = get_hf_token()
    ckpt_path, training_state_path, validation_path = download_assets(
        cache_dir, token, args.model_repo, args.checkpoint, args.data_repo, args.validation_file
    )
    breakpoints = load_breakpoints(training_state_path)
    if breakpoints is None:
        print("WARNING: easiness breakpoints unavailable. The model can still load, but target-head diagnostics need them.")

    dev = resolve_device(args.device)
    print(f"Analysis device: {dev.device} ({dev.kind})")
    model, config = load_model(ckpt_path, breakpoints, dev)
    if breakpoints is None:
        # Router target mapping requires breakpoints. Estimate them from the validation examples only
        # as a fallback; this is explicitly less faithful than training_state.json.
        vals = pq.read_table(str(validation_path), columns=["easiness_score"]).column("easiness_score").to_numpy(zero_copy_only=False)
        vals = np.asarray(vals, dtype=np.float64)
        vals = vals[np.isfinite(vals)]
        breakpoints = np.quantile(vals, np.linspace(0.0, 1.0, 101)).tolist()
        model.config.easiness_cdf_breakpoints = breakpoints
        for block in model.model.blocks:
            block.mlt_vw_rtr.config.easiness_cdf_breakpoints = breakpoints
        print("Fallback: estimated 101 easiness breakpoints from validation data.")

    batches = prepare_batches(
        validation_path, config, args.num_examples, args.batch_size, args.seq_len, args.seed
    )

    print("\n[1/5] Router/cardinality statistics...")
    router_stats = collect_router_statistics(model, batches, dev, output_dir, selected_layers)
    print(f"  Routed diagnostic CE: {router_stats['routed_ce']:.6f}")

    print("\n[2/5] Controlled CE comparisons...")
    ce_modes = {
        "routed": router_stats["routed_ce"],
        "forced_dense": evaluate_override_mode(model, batches, dev, "dense", args.seed),
        "permanent_only": evaluate_override_mode(model, batches, dev, "permanent_only", args.seed),
    }
    random_vals = []
    for trial in range(args.random_trials):
        v = evaluate_override_mode(model, batches, dev, "random_same_count", args.seed + 1000 + trial)
        random_vals.append(v)
        print(f"  Random same-count trial {trial}: {v:.6f}")
    ce_modes["random_same_count_mean"] = float(np.mean(random_vals))
    ce_modes["random_same_count_std"] = float(np.std(random_vals))
    print("  CE modes:", ce_modes)

    print("\n[3/5] Router vs CE-utility alignment (temporary gates; no parameter updates)...")
    utility = utility_alignment_analysis(
        model, batches, dev, min(args.utility_batches, len(batches)), output_dir
    )
    print(
        f"  subset routed={utility['subset_routed_ce']:.6f} | "
        f"dense={utility['dense_gate_ce']:.6f} | "
        f"signed first-order oracle same-count={utility['first_order_signed_oracle_same_count_ce']:.6f}"
    )

    print("\n[4/5] Functional + parameter redundancy...")
    redundancy = functional_redundancy_analysis(
        model, batches, dev, selected_layers,
        min(args.functional_batches, len(batches)),
        args.functional_sample_tokens,
        output_dir,
    )
    for li, r in redundancy.items():
        print(
            f"  L{li}: functional effective rank={r['functional_entropy_effective_rank']:.2f}/32, "
            f"mean |cos|={r['functional_similarity']['mean_abs']:.3f}"
        )

    exact = None
    if args.exact_ablation:
        print("\n[5/5] Exact single-head dense ablation (slow)...")
        exact = exact_ablation_analysis(
            model, batches, dev, selected_layers,
            min(args.exact_ablation_batches, len(batches)), output_dir
        )
    else:
        print("\n[5/5] Exact single-head ablation skipped (use --exact-ablation to enable).")

    summary_payload = {
        "args": vars(args),
        "checkpoint": f"{args.model_repo}/{args.checkpoint}",
        "ce_modes": ce_modes,
        "router_per_layer": router_stats["per_layer"],
        "utility": utility,
        "redundancy": redundancy,
        "exact_ablation": exact,
    }
    with (output_dir / "summary.json").open("w") as f:
        json.dump(json_safe(summary_payload), f, indent=2)

    write_summary(output_dir, args, router_stats, ce_modes, utility, redundancy, exact)

    # IMPORTANT: create archive OUTSIDE output_dir.
    # Otherwise shutil can recursively zip the ZIP while it is being written.
    archive_base = output_dir.parent / f"{output_dir.name}_results"
    
    archive_path = shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=output_dir,
    )
    print("\n" + "=" * 80)
    print("ANALYSIS COMPLETE")
    print(f"Summary: {output_dir / 'summary.md'}")
    print(f"JSON:    {output_dir / 'summary.json'}")
    print(f"ZIP:     {archive_path}")
    print("Send me summary.md + the ZIP (or at least summary.json and the CSVs) and I can interpret it.")
    print("=" * 80)


if __name__ == "__main__":
    main()

Overwriting analyze_helm7c_heads.py


In [7]:
!python analyze_helm7c_heads.py \
    --device xla \
    --batch-size 2 \
    --num-examples 8 \
    --utility-batches 4 \
    --functional-batches 4 \
    --exact-ablation \
    --exact-ablation-batches 2

HELM_7c HEAD DYNAMICS ANALYSIS
This is a read-only checkpoint analysis. Model parameters will not be updated.

Loaded 101 easiness CDF breakpoints from training_state.json
/kaggle/working/analyze_helm7c_heads.py:344: DeprecationWarning: Use torch_xla.device instead
  return DeviceContext(xm.xla_device(), "xla", xm=xm, autocast_dtype=torch.bfloat16)
E0000 00:00:1786397159.599747    1799 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238
Analysis device: xla:0 (xla)
Loading checkpoint from /kaggle/working/helm7c_analysis_cache/model_repo/checkpoint-006500.pt
Prepared 8 examples -> 4 batches of 2, seq_len=1024

[1/5] Router/cardinality statistics...
/kaggle/working/analyze_helm7c_heads.py:312: DeprecationWarning: Use torch_xla.sync instead
  self.xm.mark_step()
  Routed diagnostic CE: 3.558